# Notebook 07 — V1 Localization Audit and Early-Warning Readiness

Corrected notebook for fresh Google Colab execution. Validates the 8-input/1-output future-SOH contract, uses the explicitly selected B3 checkpoint and train-only V2 scaler, computes localization metrics separately for each horizon, and reports missing cell coverage honestly.

**Important:** actual all-100-cell coverage and measured early-warning lead time depend on source data. The notebook does not fabricate missing cells or infer lead time from prediction horizon. It will report unmet requirements if the data does not support them.

In [1]:
# CELL 1 — Environment, Drive, seed
import os, sys, json, random, gc, platform, ast, re
from pathlib import Path
import numpy as np, pandas as pd, torch
SEED=20261012
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    raise RuntimeError("Run this notebook in Google Colab and authorize Google Drive.") from e
ROOT=Path("/content/drive/MyDrive/PINN_EV_Battery")
if not ROOT.exists(): raise FileNotFoundError(f"Project folder not found: {ROOT}. Update ROOT in Cell 1 if your folder name differs.")
print("Python:",sys.version.split()[0],"| PyTorch:",torch.__version__,"| Device:",DEVICE)
print("GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("Project:",ROOT)

Mounted at /content/drive
Python: 3.13.15 | PyTorch: 2.11.0+cu130 | Device: cuda
GPU: Tesla T4
Project: /content/drive/MyDrive/PINN_EV_Battery


In [2]:
# CELL 2 — Immutable model contract and output paths
INPUT_COLS=["cell_voltage_V","cell_current_A","cell_temperature_K","surface_temperature_K",
            "soc_proxy","requested_charge_rate_C","ambient_temperature_C","horizon_s"]
TARGET_COL="capacity_retention_soh__future"
HORIZONS={60,300,600}
DATA=ROOT/"data"/"v1"
OUT=ROOT/"results_v1_localization"
OUT.mkdir(parents=True,exist_ok=True)
assert len(INPUT_COLS)==8 and TARGET_COL=="capacity_retention_soh__future"
def existing(paths): return next((Path(x) for x in paths if Path(x).exists()),None)
cell_path=existing([DATA/"cell_timeseries.csv",DATA/"cell_timeseries.parquet"])
scenario_path=existing([DATA/"scenario_instances.csv",DATA/"scenario_instances.parquet"])
sup_path=existing([DATA/"06_supervised_future_table.csv",DATA/"06_supervised_future_table.parquet"])
print("Contract: 8 inputs -> 1 target; horizons:",sorted(HORIZONS))
print("Cell data:",cell_path,"\nScenarios:",scenario_path,"\nSupervised table:",sup_path)

Contract: 8 inputs -> 1 target; horizons: [60, 300, 600]
Cell data: /content/drive/MyDrive/PINN_EV_Battery/data/v1/cell_timeseries.csv 
Scenarios: /content/drive/MyDrive/PINN_EV_Battery/data/v1/scenario_instances.csv 
Supervised table: /content/drive/MyDrive/PINN_EV_Battery/data/v1/06_supervised_future_table.csv


In [3]:

# ============================================================
# CELL 3 — AUTHORITATIVE ARTIFACT INVENTORY AND CHECKPOINT GATE
# ============================================================

import json
import torch
import pandas as pd
from pathlib import Path

print("=" * 88)
print("CELL 3 — AUTHORITATIVE CHECKPOINT COMPATIBILITY")
print("=" * 88)

# ------------------------------------------------------------
# 1. VALIDATE PREREQUISITES
# ------------------------------------------------------------

required_names = ["ROOT", "OUT", "INPUT_COLS", "TARGET_COL"]

missing = [name for name in required_names if name not in globals()]

if missing:
    raise RuntimeError(
        "STOP — run the project configuration cell first. "
        f"Missing variables: {missing}"
    )

ROOT = Path(ROOT)
OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)

INPUT_COLS = list(INPUT_COLS)
TARGET_COL = str(TARGET_COL)

EXPECTED_INPUTS = [
    "cell_voltage_V",
    "cell_current_A",
    "cell_temperature_K",
    "surface_temperature_K",
    "soc_proxy",
    "requested_charge_rate_C",
    "ambient_temperature_C",
    "horizon_s",
]

EXPECTED_TARGET = "capacity_retention_soh__future"

if INPUT_COLS != EXPECTED_INPUTS:
    raise RuntimeError(
        "STOP — INPUT_COLS does not match the authoritative V1 contract.\n"
        f"Actual:   {INPUT_COLS}\n"
        f"Expected: {EXPECTED_INPUTS}"
    )

if TARGET_COL != EXPECTED_TARGET:
    raise RuntimeError(
        "STOP — target mismatch.\n"
        f"Actual:   {TARGET_COL}\n"
        f"Expected: {EXPECTED_TARGET}"
    )

# ------------------------------------------------------------
# 2. EXPLICIT CHECKPOINT SELECTION
# ------------------------------------------------------------
# Do not choose the first matching checkpoint from a directory scan.
# Use the 8-input B3 domain-informed model identified in the
# supplied checkpoint inventory.
#
# If the scientific experiment is intended to use B1 instead,
# change this path explicitly and document the model choice.

CHECKPOINT = (
    ROOT
    / "data"
    / "v1"
    / "results_v1"
    / "models_v2"
    / "B3_physics_domain_v2_best.pt"
)

if not CHECKPOINT.is_file():
    raise FileNotFoundError(
        "STOP — selected B3 checkpoint was not found:\n"
        f"{CHECKPOINT}\n"
        "Inspect the artifact inventory. Do not substitute a legacy "
        "checkpoint automatically."
    )

# ------------------------------------------------------------
# 3. INVENTORY CHECKPOINTS
# ------------------------------------------------------------

all_files = [p for p in ROOT.rglob("*") if p.is_file()]

checkpoint_files = [
    p for p in all_files
    if p.suffix.lower() in {".pt", ".pth"}
]

inventory = []

for path in checkpoint_files:
    inventory.append({
        "path": str(path),
        "bytes": path.stat().st_size,
        "selected": path.resolve() == CHECKPOINT.resolve(),
    })

inventory_path = OUT / "07_artifact_inventory.csv"

pd.DataFrame(inventory).to_csv(
    inventory_path,
    index=False,
)

print("Project root:", ROOT)
print("Selected model:", CHECKPOINT)
print("Checkpoint inventory:", inventory_path)

# ------------------------------------------------------------
# 4. LOAD AND INSPECT SELECTED CHECKPOINT
# ------------------------------------------------------------

def load_checkpoint_file(path):
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(path, map_location="cpu")


def extract_state_dict(obj):
    if not isinstance(obj, dict):
        raise RuntimeError(
            "Unsupported checkpoint format: expected a dictionary."
        )

    for key in (
        "model_state_dict",
        "state_dict",
        "network_state_dict",
    ):
        if isinstance(obj.get(key), dict):
            return obj[key]

    if obj and all(torch.is_tensor(v) for v in obj.values()):
        return obj

    raise RuntimeError(
        "Could not locate a model state_dict in the checkpoint."
    )


def extract_metadata(obj, keys):
    if not isinstance(obj, dict):
        return None

    for key in keys:
        value = obj.get(key)
        if value is not None:
            return value

    return None


def inspect_model_dimensions(state_dict):
    weights = [
        (key, value)
        for key, value in state_dict.items()
        if key.endswith("weight")
        and torch.is_tensor(value)
        and value.ndim == 2
    ]

    if not weights:
        raise RuntimeError(
            "No two-dimensional layer weights found."
        )

    # Use the actual parameter names and dimensions.
    first_name, first_weight = weights[0]
    last_name, last_weight = weights[-1]

    return {
        "input_dim": int(first_weight.shape[1]),
        "output_dim": int(last_weight.shape[0]),
        "first_layer": first_name,
        "last_layer": last_name,
    }


try:
    CKPT = load_checkpoint_file(CHECKPOINT)
    STATE = extract_state_dict(CKPT)
    dimensions = inspect_model_dimensions(STATE)

    metadata_inputs = extract_metadata(
        CKPT,
        ["input_columns", "input_cols"],
    )

    metadata_outputs = extract_metadata(
        CKPT,
        ["output_columns", "output_cols"],
    )

    inspection = {
        "path": str(CHECKPOINT),
        "dimensions": dimensions,
        "input_columns": metadata_inputs,
        "output_columns": metadata_outputs,
        "expected_input_columns": EXPECTED_INPUTS,
        "expected_output_columns": [EXPECTED_TARGET],
    }

    # Write both the descriptive filename and the canonical filename used by the final gate.
    inspection_text = json.dumps(inspection, indent=2, default=str)
    (OUT / "07_selected_checkpoint_inspection.json").write_text(
        inspection_text, encoding="utf-8"
    )
    (OUT / "07_checkpoint_inspection.json").write_text(
        inspection_text, encoding="utf-8"
    )

except Exception as exc:
    raise RuntimeError(
        "STOP — selected checkpoint could not be inspected safely.\n"
        f"Path: {CHECKPOINT}\n"
        f"Reason: {exc}"
    ) from exc

print()
print("Checkpoint dimensions:", dimensions)
print("Checkpoint input metadata:", metadata_inputs)
print("Checkpoint output metadata:", metadata_outputs)

# ------------------------------------------------------------
# 5. STRICT COMPATIBILITY GATE
# ------------------------------------------------------------

if dimensions["input_dim"] != 8:
    raise RuntimeError(
        "CHECKPOINT COMPATIBILITY: FAIL\n"
        f"Expected 8 inputs; found {dimensions['input_dim']}."
    )

if dimensions["output_dim"] != 1:
    raise RuntimeError(
        "CHECKPOINT COMPATIBILITY: FAIL\n"
        f"Expected 1 output; found {dimensions['output_dim']}."
    )

if metadata_inputs != EXPECTED_INPUTS:
    raise RuntimeError(
        "CHECKPOINT COMPATIBILITY: FAIL\n"
        "Embedded input metadata is missing or incompatible.\n"
        f"Expected: {EXPECTED_INPUTS}\n"
        f"Actual:   {metadata_inputs}\n"
        "Do not infer missing metadata from tensor dimensions alone."
    )

if metadata_outputs != [EXPECTED_TARGET]:
    raise RuntimeError(
        "CHECKPOINT COMPATIBILITY: FAIL\n"
        "Embedded output metadata is missing or incompatible.\n"
        f"Expected: {[EXPECTED_TARGET]}\n"
        f"Actual:   {metadata_outputs}"
    )

print()
print("Checkpoint input dimension: PASS")
print("Checkpoint output dimension: PASS")
print("Checkpoint input metadata: PASS")
print("Checkpoint output metadata: PASS")
print("CHECKPOINT COMPATIBILITY: PASS")
print()
print("Selected checkpoint:")
print(CHECKPOINT)
print("=" * 88)
print("CELL 3: PASS")
print("=" * 88)


CELL 3 — AUTHORITATIVE CHECKPOINT COMPATIBILITY
Project root: /content/drive/MyDrive/PINN_EV_Battery
Selected model: /content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/models_v2/B3_physics_domain_v2_best.pt
Checkpoint inventory: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_artifact_inventory.csv

Checkpoint dimensions: {'input_dim': 8, 'output_dim': 1, 'first_layer': 'network.0.weight', 'last_layer': 'network.6.weight'}
Checkpoint input metadata: ['cell_voltage_V', 'cell_current_A', 'cell_temperature_K', 'surface_temperature_K', 'soc_proxy', 'requested_charge_rate_C', 'ambient_temperature_C', 'horizon_s']
Checkpoint output metadata: ['capacity_retention_soh__future']

Checkpoint input dimension: PASS
Checkpoint output dimension: PASS
Checkpoint input metadata: PASS
Checkpoint output metadata: PASS
CHECKPOINT COMPATIBILITY: PASS

Selected checkpoint:
/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/models_v2/B3_physics_domain_v2_best.pt
CELL 3: PASS


In [4]:
# CELL 4 — Load and validate the exact V2 train-only normalization artifact
import json
import numpy as np
from pathlib import Path

SCALER = ROOT / "data" / "v1" / "results_v1" / "06_training_normalization_v2.json"
EXPECTED_INPUTS = [
    "cell_voltage_V", "cell_current_A", "cell_temperature_K",
    "surface_temperature_K", "soc_proxy", "requested_charge_rate_C",
    "ambient_temperature_C", "horizon_s"
]
EXPECTED_TARGET = "capacity_retention_soh__future"

if not SCALER.is_file():
    raise FileNotFoundError(f"V2 normalization artifact not found: {SCALER}")

with open(SCALER, "r", encoding="utf-8") as f:
    norm = json.load(f)

stats = {}
errors = []
if norm.get("input_columns") != EXPECTED_INPUTS:
    errors.append("input_columns mismatch")
if norm.get("output_columns") != [EXPECTED_TARGET]:
    errors.append("output_columns mismatch")
if str(norm.get("normalization_split", "")).strip().lower() != "train_only":
    errors.append("normalization_split is not train_only")

try:
    X_mean = np.asarray(norm["x_mean"], dtype=np.float32).reshape(-1)
    X_std  = np.asarray(norm["x_std"], dtype=np.float32).reshape(-1)
    Y_mean = np.asarray(norm["y_mean"], dtype=np.float32).reshape(-1)
    Y_std  = np.asarray(norm["y_std"], dtype=np.float32).reshape(-1)
    stats = {"X_mean": X_mean, "X_std": X_std, "Y_mean": Y_mean, "Y_std": Y_std}
except Exception as e:
    errors.append(f"normalization statistics missing/unreadable: {e}")

if stats:
    if X_mean.shape != (8,) or X_std.shape != (8,):
        errors.append("X statistics must have shape (8,)")
    if Y_mean.shape != (1,) or Y_std.shape != (1,):
        errors.append("Y statistics must have shape (1,)")
    if not all(np.isfinite(a).all() for a in stats.values()):
        errors.append("statistics contain NaN/Inf")
    if np.any(X_std <= 0) or np.any(Y_std <= 0):
        errors.append("standard deviations must be positive")

norm_audit = {
    "path": str(SCALER),
    "compatible": len(errors) == 0,
    "input_columns": norm.get("input_columns"),
    "output_columns": norm.get("output_columns"),
    "normalization_split": norm.get("normalization_split"),
    "train_only_statistics": norm.get("train_only_statistics"),
    "shapes": {k: list(v.shape) for k, v in stats.items()},
    "errors": errors,
}
(OUT / "07_normalization_inspection.json").write_text(
    json.dumps([norm_audit], indent=2, default=str), encoding="utf-8"
)

if errors:
    raise RuntimeError("NORMALIZATION COMPATIBILITY: FAIL — " + "; ".join(errors))

print("NORMALIZATION COMPATIBILITY: PASS")
print("Artifact:", SCALER)
print("X_mean / X_std:", X_mean.shape, X_std.shape)
print("Y_mean / Y_std:", Y_mean.shape, Y_std.shape)
print("Fit split:", norm["normalization_split"])


NORMALIZATION COMPATIBILITY: PASS
Artifact: /content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/06_training_normalization_v2.json
X_mean / X_std: (8,) (8,)
Y_mean / Y_std: (1,) (1,)
Fit split: train_only


In [5]:
# CELL 5 — Validate dataset schemas (lightweight reads only)
if cell_path is None or scenario_path is None or sup_path is None:
    raise FileNotFoundError("Required Notebook 04/05/06 artifacts missing. Restore them; do not regenerate or overwrite V1.")
def head(p,n=10): return pd.read_parquet(p) if p.suffix==".parquet" else pd.read_csv(p,nrows=n)
ch=head(cell_path); sh=head(scenario_path); uh=head(sup_path)
need_cell=["scenario_instance_id","scenario_id","cell_id","time_s"]+[c for c in INPUT_COLS if c!="horizon_s"]
need_scenario=["scenario_instance_id","scenario_id","fault_cells_json"]
need_sup=[c for c in INPUT_COLS]+[TARGET_COL]
missing={"cell":sorted(set(need_cell)-set(ch.columns)),"scenario":sorted(set(need_scenario)-set(sh.columns)),"supervised":sorted(set(need_sup)-set(uh.columns))}
(OUT/"07_dataset_schema_audit.json").write_text(json.dumps(missing,indent=2))
if any(missing.values()): raise RuntimeError(f"Dataset schema mismatch: {missing}")
if any("fault" in x.lower() for x in INPUT_COLS): raise RuntimeError("Fault-label leakage in input contract.")
print("Dataset schemas PASS. Fault-location labels are not model inputs.")

Dataset schemas PASS. Fault-location labels are not model inputs.


In [6]:
# CELL 6 — Reconstruct strict model; no non-strict loading
import torch.nn as nn
class PredictiveMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network=nn.Sequential(nn.Linear(8,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,1))
    def forward(self,x): return self.network(x)
model=PredictiveMLP()
try: model.load_state_dict(STATE,strict=True)
except Exception as e: raise RuntimeError("Checkpoint names/shapes do not match the declared 8→1 width-128 depth-3 MLP. Do not use strict=False; reconcile with the executed training contract.") from e
model.to(DEVICE).eval()
with torch.no_grad(): probe=model(torch.zeros(3,8,device=DEVICE))
if tuple(probe.shape)!=(3,1): raise RuntimeError(f"Forward contract failed: {tuple(probe.shape)}")
print("Model load PASS; output shape:",tuple(probe.shape))

Model load PASS; output shape: (3, 1)


In [7]:
# CELL 7 — Validate scenario split isolation
split_path=existing([DATA/"05_split_integrity_summary.csv",DATA/"05_split_manifest.csv",DATA/"scenario_split_manifest.csv",ROOT/"results"/"05_split_integrity_summary.csv"])
if split_path is None: raise FileNotFoundError("Notebook 05 split manifest not found. Restore the authoritative manifest; do not create a new split after test results.")
spl=pd.read_csv(split_path)
ic=next((c for c in ["scenario_instance_id","instance_id"] if c in spl.columns),None)
sc=next((c for c in ["data_split","split","dataset_split"] if c in spl.columns),None)
if not ic or not sc: raise RuntimeError(f"Split manifest columns unexpected: {list(spl.columns)}")
spl[ic]=spl[ic].astype(str); spl[sc]=spl[sc].astype(str).str.lower()
if (spl.groupby(ic)[sc].nunique()>1).any(): raise RuntimeError("Scenario-instance leakage: instance occurs in multiple splits.")
split_map=dict(zip(spl[ic],spl[sc]))
(OUT/"07_split_isolation_audit.json").write_text(json.dumps({"manifest":str(split_path),"instances":int(spl[ic].nunique()),"split_counts":spl[sc].value_counts().to_dict(),"isolation":True},indent=2))
print("Scenario-instance isolation PASS.")

Scenario-instance isolation PASS.


In [8]:
# CELL 8 — Safe batched inference function
def predict_future_soh(raw):
    raw=np.asarray(raw,dtype=np.float32)
    if raw.ndim!=2 or raw.shape[1]!=8: raise ValueError(f"Expected (N,8), got {raw.shape}")
    if not np.isfinite(raw).all(): raise ValueError("Input contains NaN/Inf.")
    x=(raw-X_mean.astype(np.float32))/X_std.astype(np.float32)
    out=[]
    with torch.no_grad():
        for i in range(0,len(x),65536):
            out.append(model(torch.from_numpy(x[i:i+65536]).to(DEVICE)).cpu().numpy())
    y=np.concatenate(out)*Y_std.astype(np.float32)+Y_mean.astype(np.float32)
    if not np.isfinite(y).all(): raise RuntimeError("Model predictions contain NaN/Inf.")
    return y.reshape(-1)
print("Batched inference ready.")

Batched inference ready.


## Important interpretation

The model output is future SOH, not a fault class. We therefore rank cells using deviation from the median predicted future SOH within the same scenario instance, time, and horizon. Ground-truth fault labels are joined **only after** predictions/scores are saved. This is a transparent exploratory heuristic, not a validated safety alarm.

In [9]:

# CELL 9 — Complete test inference without per-instance row truncation

import json
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path

BASE = [c for c in INPUT_COLS if c != "horizon_s"]
PRED_PATH = OUT / "07_cell_predictions.csv"

if not cell_path.exists():
    raise FileNotFoundError(f"Cell dataset not found: {cell_path}")

# Start clean so old predictions cannot contaminate this run.
if PRED_PATH.exists():
    PRED_PATH.unlink()

# Coverage sets are collected from the actual source, not inferred.
source_cells = set()
test_split_cells = set()
valid_test_cells = set()

source_rows = 0
test_rows = 0
valid_test_rows = 0
rows_written = 0

def score_frame(frame):
    global source_rows, test_rows, valid_test_rows, rows_written

    d = frame.copy()
    source_rows += len(d)

    required = (
        ["scenario_instance_id", "scenario_id", "cell_id", "time_s"]
        + BASE
    )
    missing_columns = [c for c in required if c not in d.columns]
    if missing_columns:
        raise RuntimeError(
            f"Dataset is missing required columns: {missing_columns}"
        )

    # Record cell coverage before filtering features or rows.
    raw_cells = pd.to_numeric(d["cell_id"], errors="coerce")
    source_cells.update(
        raw_cells.dropna().astype(int).tolist()
    )

    d["scenario_instance_id"] = d["scenario_instance_id"].astype(str)

    split_values = (
        d["scenario_instance_id"]
        .map(split_map)
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    d = d.loc[split_values.eq("test")].copy()
    if d.empty:
        return 0

    test_rows += len(d)

    test_cell_values = pd.to_numeric(d["cell_id"], errors="coerce")
    test_split_cells.update(
        test_cell_values.dropna().astype(int).tolist()
    )

    # Validate numeric inputs without discarding entire cells unnecessarily.
    for c in BASE + ["time_s", "cell_id"]:
        d[c] = pd.to_numeric(d[c], errors="coerce")

    d = d.dropna(subset=BASE + ["time_s", "cell_id"])

    # Cell IDs must be integer-valued.
    d = d[np.isclose(d["cell_id"], np.round(d["cell_id"]))]
    d["cell_id"] = d["cell_id"].astype(int)

    finite_mask = np.isfinite(
        d[BASE].to_numpy(dtype=np.float64)
    ).all(axis=1)
    d = d.loc[finite_mask].copy()

    if d.empty:
        return 0

    valid_test_rows += len(d)
    valid_test_cells.update(d["cell_id"].astype(int).tolist())

    # Predict every valid held-out row at every requested horizon.
    # No per-instance row cap or sampling is applied.
    output_parts = []

    base_values = d[BASE].to_numpy(dtype=np.float32)

    for h in sorted(HORIZONS):
        x = np.column_stack([
            base_values,
            np.full(len(d), h, dtype=np.float32)
        ])

        y = np.asarray(predict_future_soh(x)).reshape(-1)

        if len(y) != len(d):
            raise RuntimeError(
                f"Prediction length mismatch at horizon {h}: "
                f"{len(y)} predictions for {len(d)} rows."
            )

        if not np.isfinite(y).all():
            raise RuntimeError(
                f"Non-finite predictions detected at horizon {h}."
            )

        z = d[
            ["scenario_instance_id", "scenario_id", "cell_id", "time_s"]
        ].copy()

        z["horizon_s"] = int(h)
        z["predicted_future_soh"] = y
        output_parts.append(z)

    result = pd.concat(output_parts, ignore_index=True)

    result.to_csv(
        PRED_PATH,
        mode="a",
        header=not PRED_PATH.exists(),
        index=False
    )

    rows_written += len(result)
    return len(result)


# Process CSV or Parquet incrementally.
if cell_path.suffix.lower() == ".csv":
    for chunk in pd.read_csv(cell_path, chunksize=100000):
        score_frame(chunk)
        del chunk
        gc.collect()

elif cell_path.suffix.lower() in {".parquet", ".pq"}:
    parquet_file = pq.ParquetFile(cell_path)

    for batch in parquet_file.iter_batches(batch_size=100000):
        chunk = batch.to_pandas()
        score_frame(chunk)
        del chunk, batch
        gc.collect()

else:
    raise ValueError(f"Unsupported dataset format: {cell_path.suffix}")

if rows_written == 0 or not PRED_PATH.exists():
    raise RuntimeError(
        "No held-out test predictions were generated. "
        "Check split_map and scenario_instance_id values."
    )

# Load generated predictions for downstream localization.
pred = pd.read_csv(PRED_PATH)

pred["scenario_instance_id"] = pred["scenario_instance_id"].astype(str)
pred["cell_id"] = pd.to_numeric(
    pred["cell_id"], errors="raise"
).astype(int)
pred["horizon_s"] = pd.to_numeric(
    pred["horizon_s"], errors="raise"
).astype(int)

observed_prediction_cells = sorted(
    pred["cell_id"].unique().astype(int).tolist()
)

expected_cells = list(range(1, 101))

source_missing = sorted(set(expected_cells) - source_cells)
test_missing = sorted(set(expected_cells) - test_split_cells)
valid_missing = sorted(set(expected_cells) - valid_test_cells)
prediction_missing = sorted(
    set(expected_cells) - set(observed_prediction_cells)
)

coverage = {
    "expected_cell_ids": expected_cells,
    "source_rows_scanned": int(source_rows),
    "source_unique_cell_count": len(source_cells),
    "source_observed_cell_ids": sorted(source_cells),
    "source_missing_expected_cell_ids": source_missing,
    "test_rows_before_feature_validation": int(test_rows),
    "test_split_unique_cell_count": len(test_split_cells),
    "test_split_observed_cell_ids": sorted(test_split_cells),
    "test_split_missing_expected_cell_ids": test_missing,
    "valid_test_rows_scored_before_horizons": int(valid_test_rows),
    "valid_test_unique_cell_count": len(valid_test_cells),
    "valid_test_observed_cell_ids": sorted(valid_test_cells),
    "valid_test_missing_expected_cell_ids": valid_missing,
    "prediction_rows_including_all_horizons": int(len(pred)),
    "prediction_unique_cell_count": len(observed_prediction_cells),
    "prediction_observed_cell_ids": observed_prediction_cells,
    "prediction_missing_expected_cell_ids": prediction_missing,
    "all_100_cells_in_source": set(expected_cells).issubset(source_cells),
    "all_100_cells_in_test_split": set(expected_cells).issubset(test_split_cells),
    "all_100_cells_scored": set(expected_cells).issubset(valid_test_cells),
    "all_100_cells_in_predictions": set(expected_cells).issubset(
        set(observed_prediction_cells)
    ),
    "note": (
        "Coverage is derived from source rows and valid held-out rows. "
        "No missing cells or predictions are fabricated."
    )
}

(OUT / "07_cell_coverage_audit.json").write_text(
    json.dumps(coverage, indent=2), encoding="utf-8"
)

print("SOURCE ROWS SCANNED:", source_rows)
print("SOURCE UNIQUE CELLS:", len(source_cells), sorted(source_cells))
print("TEST-SPLIT UNIQUE CELLS:", len(test_split_cells))
print("VALID TEST CELLS SCORED:", len(valid_test_cells))
print("PREDICTION ROWS:", len(pred))
print("PREDICTION UNIQUE CELLS:", len(observed_prediction_cells))
print("SOURCE MISSING CELLS:", source_missing)
print("TEST-SPLIT MISSING CELLS:", test_missing)
print("VALID-TEST MISSING CELLS:", valid_missing)
print("PREDICTION MISSING CELLS:", prediction_missing)
print("ALL 100 CELLS IN SOURCE:", coverage["all_100_cells_in_source"])
print("ALL 100 CELLS IN TEST SPLIT:", coverage["all_100_cells_in_test_split"])
print("ALL 100 CELLS SCORED:", coverage["all_100_cells_scored"])
print("Coverage report:", OUT / "07_cell_coverage_audit.json")


SOURCE ROWS SCANNED: 7656014
SOURCE UNIQUE CELLS: 100 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100]
TEST-SPLIT UNIQUE CELLS: 100
VALID TEST CELLS SCORED: 100
PREDICTION ROWS: 16452696
PREDICTION UNIQUE CELLS: 100
SOURCE MISSING CELLS: []
TEST-SPLIT MISSING CELLS: []
VALID-TEST MISSING CELLS: []
PREDICTION MISSING CELLS: []
ALL 100 CELLS IN SOURCE: True
ALL 100 CELLS IN TEST SPLIT: True
ALL 100 CELLS SCORED: True
Coverage report: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_cell_coverage_audit.json


In [10]:
# CELL 10 — Blind localization scored separately for each horizon
import ast, re, json
from pathlib import Path

pred["cohort_median"] = pred.groupby(
    ["scenario_instance_id", "time_s", "horizon_s"]
)["predicted_future_soh"].transform("median")
pred["deviation_score"] = (
    pred["predicted_future_soh"] - pred["cohort_median"]
).abs()

# One row per scenario instance × cell × horizon.
rank = pred.groupby(
    ["scenario_instance_id", "scenario_id", "cell_id", "horizon_s"],
    as_index=False
).agg(
    anomaly_score=("deviation_score", "median"),
    predicted_future_soh=("predicted_future_soh", "median"),
    time_points=("time_s", "nunique")
)
rank.to_csv(OUT / "07_localization_scores.csv", index=False)

def parse_faults(v):
    if pd.isna(v) or str(v).strip().lower() in {"", "none", "nan", "[]", "null"}:
        return []
    s = str(v)
    try:
        x = json.loads(s)
    except Exception:
        try:
            x = ast.literal_eval(s)
        except Exception:
            x = [int(q) for q in re.findall(r"\d+", s)]
    if isinstance(x, dict):
        x = x.get("fault_cells", x.get("cells", []))
    if isinstance(x, (int, float)):
        x = [int(x)]
    if not isinstance(x, (list, tuple, set)):
        raise ValueError(f"Cannot parse fault label {v!r}")
    return sorted({int(a) for a in x if str(a).lstrip("-").isdigit() and 1 <= int(a) <= 100})

scen = pd.read_parquet(scenario_path) if scenario_path.suffix.lower() == ".parquet" else pd.read_csv(scenario_path)
scen["scenario_instance_id"] = scen["scenario_instance_id"].astype(str)
faults = {
    str(r["scenario_instance_id"]): parse_faults(r["fault_cells_json"])
    for _, r in scen.iterrows()
}

metrics = []
detailed = []
healthy = []

for (inst, horizon), g0 in rank.groupby(["scenario_instance_id", "horizon_s"], sort=True):
    g = g0.sort_values(["anomaly_score", "cell_id"], ascending=[False, True]).reset_index(drop=True)
    scenario_id = str(g["scenario_id"].iloc[0])
    truth = set(faults.get(str(inst), []))
    observed = set(g["cell_id"].astype(int))
    observed_faults = truth & observed
    missing_faults = truth - observed
    n_cells = len(observed)
    k = min(max(1, len(truth)), n_cells) if n_cells else 0
    chosen_list = g.head(k)["cell_id"].astype(int).tolist()
    chosen = set(chosen_list)
    tp = len(chosen & truth)
    precision = tp / max(len(chosen), 1)
    recall = tp / len(truth) if truth else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if truth and (precision + recall) > 0 else (0.0 if truth else np.nan)
    )
    positions = {int(cell): i + 1 for i, cell in enumerate(g["cell_id"].astype(int))}
    ranks_for_observed_faults = [positions[c] for c in observed_faults if c in positions]

    metrics.append({
        "scenario_instance_id": str(inst),
        "scenario_id": scenario_id,
        "horizon_s": int(horizon),
        "n_cells_scored": n_cells,
        "ground_truth_fault_cells": json.dumps(sorted(truth)),
        "observed_fault_cells": json.dumps(sorted(observed_faults)),
        "missing_fault_cells": json.dumps(sorted(missing_faults)),
        "top_k": int(k),
        "top_k_cells": json.dumps(sorted(chosen)),
        "true_positives_at_k": int(tp),
        "precision_at_k": float(precision),
        "recall_at_k": float(recall) if np.isfinite(recall) else np.nan,
        "f1_at_k": float(f1) if np.isfinite(f1) else np.nan,
        "mean_observed_fault_rank": float(np.mean(ranks_for_observed_faults)) if ranks_for_observed_faults else np.nan,
        "fault_label_coverage": (len(observed_faults) / len(truth)) if truth else np.nan,
        "ground_truth_used_only_after_scoring": True
    })

    for _, r in g.iterrows():
        cell = int(r["cell_id"])
        detailed.append({
            "scenario_instance_id": str(inst),
            "scenario_id": scenario_id,
            "horizon_s": int(horizon),
            "cell_id": cell,
            "anomaly_score": float(r["anomaly_score"]),
            "rank": positions[cell],
            "is_fault_ground_truth": int(cell in truth)
        })

    if not truth:
        top = g.iloc[0] if not g.empty else None
        if top is not None:
            healthy.append({
                "scenario_instance_id": str(inst),
                "scenario_id": scenario_id,
                "horizon_s": int(horizon),
                "top_ranked_cell": int(top["cell_id"]),
                "score": float(top["anomaly_score"]),
                "interpretation": "relative outlier candidate only; not a validated fault alarm"
            })

m = pd.DataFrame(metrics)
det = pd.DataFrame(detailed)
if m.empty:
    raise RuntimeError("No localization metrics were produced.")

m.to_csv(OUT / "07_localization_metrics.csv", index=False)
det.to_csv(OUT / "07_localization_predictions.csv", index=False)
pd.DataFrame(healthy).to_csv(OUT / "07_healthy_case_false_localization.csv", index=False)

print("Localization metrics computed independently for each horizon.")
print(m.groupby("horizon_s")[["precision_at_k", "recall_at_k", "f1_at_k", "mean_observed_fault_rank"]].mean(numeric_only=True))
print("Metrics saved:", OUT / "07_localization_metrics.csv")


Localization metrics computed independently for each horizon.
           precision_at_k  recall_at_k  f1_at_k  mean_observed_fault_rank
horizon_s                                                                
60                    0.0          0.0      0.0                     57.25
300                   0.0          0.0      0.0                     57.25
600                   0.0          0.0      0.0                     57.25
Metrics saved: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_localization_metrics.csv


In [11]:

# CELL 10A — Diagnose zero localization metrics
import json
import numpy as np
import pandas as pd

# 1. Check scenario-instance label integrity
print("=" * 70)
print("1. GROUND-TRUTH LABEL AUDIT")
print("=" * 70)

label_counts = scen.groupby("scenario_instance_id").size()
duplicate_instances = label_counts[label_counts > 1]

print("Scenario rows:", len(scen))
print("Unique scenario instances:", scen["scenario_instance_id"].nunique())
print("Instances with duplicate scenario rows:", len(duplicate_instances))

if len(duplicate_instances):
    print("\nDuplicate scenario instances:")
    print(duplicate_instances.head(20).to_string())

print("\nFault-label examples:")
print(
    scen[["scenario_instance_id", "scenario_id", "fault_cells_json"]]
    .head(15)
    .to_string(index=False)
)

# 2. Summarize parsed fault labels
print("\n" + "=" * 70)
print("2. PARSED FAULT LABELS")
print("=" * 70)

label_summary = []
for inst, group in scen.groupby("scenario_instance_id", sort=False):
    parsed = [parse_faults(v) for v in group["fault_cells_json"]]
    union_faults = sorted(set(c for cells in parsed for c in cells))

    label_summary.append({
        "scenario_instance_id": str(inst),
        "scenario_rows": len(group),
        "fault_labels_per_row": [len(x) for x in parsed],
        "union_fault_count": len(union_faults),
        "union_fault_cells": union_faults
    })

label_audit = pd.DataFrame(label_summary)

print("Instances with at least one fault:",
      int((label_audit["union_fault_count"] > 0).sum()))
print("Instances with no parsed fault labels:",
      int((label_audit["union_fault_count"] == 0).sum()))
print("\nLabel audit sample:")
print(label_audit.head(20).to_string(index=False))

# 3. Verify rankings and fault-cell overlap by horizon
print("\n" + "=" * 70)
print("3. TOP-RANKED CELLS VS GROUND TRUTH")
print("=" * 70)

for (inst, horizon), group in rank.groupby(
    ["scenario_instance_id", "horizon_s"], sort=True
):
    inst = str(inst)

    matching_rows = scen[
        scen["scenario_instance_id"].astype(str) == inst
    ]

    truth = set()
    for value in matching_rows["fault_cells_json"]:
        truth.update(parse_faults(value))

    g = group.sort_values(
        ["anomaly_score", "cell_id"],
        ascending=[False, True]
    )

    top_k = max(1, len(truth)) if truth else 5
    top_cells = g.head(top_k)["cell_id"].astype(int).tolist()
    overlap = sorted(set(top_cells) & truth)

    fault_ranks = g.loc[
        g["cell_id"].astype(int).isin(truth)
    ].copy()

    if not fault_ranks.empty:
        fault_ranks["rank"] = (
            fault_ranks["anomaly_score"]
            .rank(method="first", ascending=False)
        )

    print({
        "instance": inst,
        "horizon_s": int(horizon),
        "fault_cells": sorted(truth),
        "top_cells": top_cells,
        "overlap": overlap,
        "fault_ranks": (
            fault_ranks["rank"].astype(int).tolist()
            if not fault_ranks.empty else []
        )
    })

# 4. Determine whether predictions/rankings change with horizon
print("\n" + "=" * 70)
print("4. HORIZON CONSISTENCY CHECK")
print("=" * 70)

for inst, group in rank.groupby("scenario_instance_id"):
    horizon_rankings = {}

    for horizon, hg in group.groupby("horizon_s"):
        ordered = hg.sort_values(
            ["anomaly_score", "cell_id"],
            ascending=[False, True]
        )["cell_id"].astype(int).tolist()

        horizon_rankings[int(horizon)] = ordered

    if len(horizon_rankings) >= 2:
        horizons = sorted(horizon_rankings)
        baseline = horizon_rankings[horizons[0]]

        for horizon in horizons[1:]:
            other = horizon_rankings[horizon]
            same = baseline == other
            print(
                f"Instance {inst}: "
                f"{horizons[0]}s vs {horizon}s "
                f"identical ranking = {same}"
            )

# 5. Check score variation across horizons
print("\n" + "=" * 70)
print("5. ANOMALY-SCORE SUMMARY BY HORIZON")
print("=" * 70)

print(
    rank.groupby("horizon_s")["anomaly_score"]
    .agg(["count", "mean", "std", "min", "median", "max"])
    .to_string()
)

print("\nDiagnostic complete. Existing Cell 10 metrics were not modified.")


1. GROUND-TRUTH LABEL AUDIT
Scenario rows: 28
Unique scenario instances: 28
Instances with duplicate scenario rows: 0

Fault-label examples:
    scenario_instance_id scenario_id fault_cells_json
              S1_HEALTHY          S1               []
              S2_HEALTHY          S2               []
              S3_HEALTHY          S3               []
              S4_CELL004          S4              [4]
              S4_CELL019          S4             [19]
              S5_CELL004          S5              [4]
              S4_CELL055          S4             [55]
              S5_CELL073          S5             [73]
        S4_CELL037_BENCH          S4             [37]
        S5_CELL037_BENCH          S5             [37]
              S4_CELL082          S4             [82]
              S5_CELL091          S5             [91]
          S5_CELL004_082          S5          [4, 82]
S6_S5_CELL037_BENCH_2p0C          S6             [37]
S6_S5_CELL037_BENCH_1p5C          S6             

In [12]:

# CELL 10B — Trace why all localization scores are zero
import numpy as np
import pandas as pd

print("=" * 65)
print("1. PREDICTION DISTRIBUTION")
print("=" * 65)

p = pd.to_numeric(pred["predicted_future_soh"], errors="coerce")

print("Prediction rows:", len(p))
print("Valid predictions:", int(p.notna().sum()))
print("Unique prediction values:", int(p.nunique()))
print("Prediction standard deviation:", float(p.std()))
print("Prediction min:", float(p.min()))
print("Prediction max:", float(p.max()))
print("Prediction sample:")
print(p.head(20).to_list())

print("\n" + "=" * 65)
print("2. VARIATION ACROSS CELLS AT EACH TIME/HORIZON")
print("=" * 65)

cohort_keys = ["scenario_instance_id", "time_s", "horizon_s"]

cohort_stats = (
    pred.groupby(cohort_keys, dropna=False)["predicted_future_soh"]
    .agg(["count", "nunique", "std", "min", "max"])
    .reset_index()
)

print("Cohort groups:", len(cohort_stats))
print("Groups with >1 unique prediction:",
      int((cohort_stats["nunique"] > 1).sum()))
print("Groups with exactly 1 unique prediction:",
      int((cohort_stats["nunique"] == 1).sum()))
print("Maximum prediction spread within a cohort:",
      float((cohort_stats["max"] - cohort_stats["min"]).max()))

print("\nExample cohort statistics:")
print(cohort_stats.head(10).to_string(index=False))

print("\n" + "=" * 65)
print("3. PREDICTION VARIATION BY HORIZON")
print("=" * 65)

print(
    pred.groupby("horizon_s")["predicted_future_soh"]
    .agg(["count", "nunique", "mean", "std", "min", "max"])
    .to_string()
)

print("\n" + "=" * 65)
print("4. PREDICTION VARIATION BY CELL")
print("=" * 65)

cell_stats = (
    pred.groupby("cell_id")["predicted_future_soh"]
    .agg(["count", "nunique", "mean", "std", "min", "max"])
)

print("Cells with varying predictions:",
      int((cell_stats["nunique"] > 1).sum()), "of", len(cell_stats))
print("Cell prediction summary:")
print(cell_stats.head(15).to_string())

print("\n" + "=" * 65)
print("5. AVAILABLE PREDICTION COLUMNS")
print("=" * 65)

print(list(pred.columns))

# Check whether original model input columns were saved in predictions.
input_cols_present = [
    c for c in INPUT_COLS if c in pred.columns
]
print("Model input columns available in pred:", input_cols_present)

if input_cols_present:
    print("\nInput variation by feature:")
    for col in input_cols_present:
        values = pd.to_numeric(pred[col], errors="coerce")
        print({
            "column": col,
            "unique_values": int(values.nunique()),
            "std": float(values.std()) if values.notna().any() else None
        })

print("\n" + "=" * 65)
print("6. HORIZON-SPECIFIC RANK SCORE VARIATION")
print("=" * 65)

print(
    rank.groupby("horizon_s")["anomaly_score"]
    .agg(["count", "nunique", "mean", "std", "min", "max"])
    .to_string()
)

print("\nDiagnostic complete. No predictions or metrics were changed.")


1. PREDICTION DISTRIBUTION
Prediction rows: 16452696
Valid predictions: 16452696
Unique prediction values: 1403
Prediction standard deviation: 1.422937220205706e-05
Prediction min: 0.9998583
Prediction max: 0.99995804
Prediction sample:
[0.9999172, 0.9999172, 0.9999172, 0.9999172, 0.9999172, 0.9999172, 0.99991727, 0.99991727, 0.9999173, 0.9999173, 0.9999174, 0.99991745, 0.99991757, 0.9999176, 0.9999177, 0.9999178, 0.99991786, 0.999918, 0.99991804, 0.99991816]

2. VARIATION ACROSS CELLS AT EACH TIME/HORIZON
Cohort groups: 386574
Groups with >1 unique prediction: 287
Groups with exactly 1 unique prediction: 386287
Maximum prediction spread within a cohort: 2.121999999993296e-05

Example cohort statistics:
scenario_instance_id       time_s  horizon_s  count  nunique          std      min      max
    S4_CELL037_BENCH 0.000000e+00         60    100        2 6.900000e-07 0.999910 0.999917
    S4_CELL037_BENCH 0.000000e+00        300    100        2 1.121000e-06 0.999921 0.999932
    S4_CELL

In [13]:

# CELL 10C — Find a safe timestamp alignment precision
import numpy as np
import pandas as pd

print("1. RAW TIMESTAMP PRECISION")
print("Unique raw timestamps:", pred["time_s"].nunique())
print("Timestamp min:", pred["time_s"].min())
print("Timestamp max:", pred["time_s"].max())

# Compare how many distinct cells are grouped together at different
# timestamp precisions, independently for each scenario instance.
for decimals in [7, 6, 5, 4, 3, 2, 1, 0]:
    temp = pred[
        ["scenario_instance_id", "time_s", "cell_id"]
    ].drop_duplicates().copy()

    temp["time_bucket"] = temp["time_s"].round(decimals)

    sizes = (
        temp.groupby(["scenario_instance_id", "time_bucket"])
        ["cell_id"]
        .nunique()
    )

    print(f"\nRounding to {decimals} decimal places:")
    print("  Cohort groups:", len(sizes))
    print("  Groups with 100 cells:", int((sizes == 100).sum()))
    print("  Groups with 90-99 cells:",
          int(sizes.between(90, 99).sum()))
    print("  Groups with fewer than 90 cells:",
          int((sizes < 90).sum()))
    print("  Median cells per group:", float(sizes.median()))
    print("  Minimum cells per group:", int(sizes.min()))

print("\n2. TIMESTAMP EXAMPLES BY CELL")
example_instance = pred["scenario_instance_id"].iloc[0]
example = pred[
    pred["scenario_instance_id"].astype(str) == str(example_instance)
]

for cell_id in sorted(example["cell_id"].unique())[:5]:
    times = (
        example.loc[example["cell_id"] == cell_id, "time_s"]
        .drop_duplicates()
        .sort_values()
        .head(10)
        .to_list()
    )
    print(f"Instance={example_instance}, cell={cell_id}: {times}")

print("\nDiagnostic complete; original predictions and metrics unchanged.")


1. RAW TIMESTAMP PRECISION
Unique raw timestamps: 19994
Timestamp min: 0.0
Timestamp max: 181853.7763361996

Rounding to 7 decimal places:
  Cohort groups: 127926
  Groups with 100 cells: 20
  Groups with 90-99 cells: 54445
  Groups with fewer than 90 cells: 73461
  Median cells per group: 2.0
  Minimum cells per group: 1

Rounding to 6 decimal places:
  Cohort groups: 127926
  Groups with 100 cells: 20
  Groups with 90-99 cells: 54445
  Groups with fewer than 90 cells: 73461
  Median cells per group: 2.0
  Minimum cells per group: 1

Rounding to 5 decimal places:
  Cohort groups: 127846
  Groups with 100 cells: 20
  Groups with 90-99 cells: 54365
  Groups with fewer than 90 cells: 73461
  Median cells per group: 2.0
  Minimum cells per group: 1

Rounding to 4 decimal places:
  Cohort groups: 127706
  Groups with 100 cells: 20
  Groups with 90-99 cells: 54225
  Groups with fewer than 90 cells: 73461
  Median cells per group: 2.0
  Minimum cells per group: 1

Rounding to 3 decimal place

In [14]:

# CELL 10D — Check whether cells share a common simulation time grid
import numpy as np
import pandas as pd

# Use one scenario instance and one horizon to avoid mixing runs.
example_instance = str(pred["scenario_instance_id"].iloc[0])
example_horizon = int(pred["horizon_s"].iloc[0])

sample = pred[
    (pred["scenario_instance_id"].astype(str) == example_instance)
    & (pred["horizon_s"] == example_horizon)
].copy()

sample["time_s"] = pd.to_numeric(sample["time_s"], errors="coerce")
sample["cell_id"] = pd.to_numeric(sample["cell_id"], errors="coerce")
sample = sample.dropna(subset=["time_s", "cell_id"])

print("Scenario instance:", example_instance)
print("Horizon:", example_horizon)
print("Cells present:", sample["cell_id"].nunique())

# Count unique timestamps and inspect time-step distributions per cell.
print("\n1. TIME GRID SUMMARY")
for cell_id in [1, 4, 19, 37, 55, 82, 91]:
    cell = sample[sample["cell_id"] == cell_id]
    times = np.sort(cell["time_s"].unique())
    if len(times) < 2:
        print(cell_id, "insufficient timestamps:", len(times))
        continue

    dt = np.diff(times)
    print({
        "cell_id": cell_id,
        "unique_times": len(times),
        "first_time_s": float(times[0]),
        "last_time_s": float(times[-1]),
        "median_step_s": float(np.median(dt)),
        "min_step_s": float(np.min(dt)),
        "max_step_s": float(np.max(dt))
    })

# Compare timestamps with cell 1, using nearest-neighbor matching.
print("\n2. ALIGNMENT TO CELL 1")
reference = np.sort(
    sample.loc[sample["cell_id"] == 1, "time_s"].unique()
)

if len(reference) == 0:
    print("Cell 1 has no timestamps in this scenario/horizon.")
else:
    for cell_id in sorted(sample["cell_id"].astype(int).unique())[:20]:
        times = np.sort(
            sample.loc[sample["cell_id"] == cell_id, "time_s"].unique()
        )
        if len(times) == 0:
            continue

        idx = np.searchsorted(reference, times)
        right = np.clip(idx, 0, len(reference) - 1)
        left = np.clip(idx - 1, 0, len(reference) - 1)

        distance = np.minimum(
            np.abs(times - reference[right]),
            np.abs(times - reference[left])
        )

        print({
            "cell_id": int(cell_id),
            "timestamp_count": len(times),
            "fraction_within_1e-4_s": float(np.mean(distance <= 1e-4)),
            "fraction_within_1e-2_s": float(np.mean(distance <= 1e-2)),
            "fraction_within_0.5_s": float(np.mean(distance <= 0.5)),
            "median_nearest_difference_s": float(np.median(distance))
        })

print("\n3. SAMPLE TIMESTAMPS")
for cell_id in [1, 4, 19, 37, 55, 82, 91]:
    times = np.sort(
        sample.loc[sample["cell_id"] == cell_id, "time_s"].unique()
    )
    print(f"Cell {cell_id}:", times[:15].tolist())

print("\nDiagnostic only: no predictions or metrics modified.")


Scenario instance: S4_CELL037_BENCH
Horizon: 60
Cells present: 100

1. TIME GRID SUMMARY
{'cell_id': 1, 'unique_times': 2690, 'first_time_s': 0.0, 'last_time_s': 7227.598871386735, 'median_step_s': 2.4303848318175056, 'min_step_s': 2.220446049250313e-16, 'max_step_s': 28.44069098606724}
{'cell_id': 4, 'unique_times': 2690, 'first_time_s': 0.0, 'last_time_s': 7227.598871386735, 'median_step_s': 2.4303848318175056, 'min_step_s': 2.220446049250313e-16, 'max_step_s': 28.44069098606724}
{'cell_id': 19, 'unique_times': 2690, 'first_time_s': 0.0, 'last_time_s': 7227.598871386735, 'median_step_s': 2.4303848318175056, 'min_step_s': 2.220446049250313e-16, 'max_step_s': 28.44069098606724}
{'cell_id': 37, 'unique_times': 3681, 'first_time_s': 0.0, 'last_time_s': 178170.49763841563, 'median_step_s': 3.0412297476723325, 'min_step_s': 9.094947017729282e-13, 'max_step_s': 300.0000000000073}
{'cell_id': 55, 'unique_times': 2690, 'first_time_s': 0.0, 'last_time_s': 7227.598871386735, 'median_step_s': 2.

In [15]:

# CELL 10E — Inspect per-cell time coverage and prediction spread
import numpy as np
import pandas as pd

instance = "S4_CELL037_BENCH"
horizon = 60

d = pred[
    (pred["scenario_instance_id"].astype(str) == instance)
    & (pred["horizon_s"] == horizon)
].copy()

d["cell_id"] = pd.to_numeric(d["cell_id"], errors="coerce")
d["time_s"] = pd.to_numeric(d["time_s"], errors="coerce")
d["predicted_future_soh"] = pd.to_numeric(
    d["predicted_future_soh"], errors="coerce"
)
d = d.dropna(subset=["cell_id", "time_s", "predicted_future_soh"])

print("1. PER-CELL COVERAGE")
coverage = (
    d.groupby("cell_id")
    .agg(
        rows=("time_s", "size"),
        unique_times=("time_s", "nunique"),
        first_time_s=("time_s", "min"),
        last_time_s=("time_s", "max"),
        prediction_std=("predicted_future_soh", "std"),
        prediction_min=("predicted_future_soh", "min"),
        prediction_max=("predicted_future_soh", "max")
    )
)

print(coverage.to_string())

print("\n2. CELL COUNTS BY TIME WINDOW")
for lower, upper in [
    (0, 100),
    (100, 1000),
    (1000, 3000),
    (3000, 7000),
    (7000, 10000),
    (10000, 30000),
    (30000, 100000),
    (100000, 181854)
]:
    part = d[d["time_s"].between(lower, upper, inclusive="left")]
    print({
        "window_s": [lower, upper],
        "rows": len(part),
        "cells_present": int(part["cell_id"].nunique()),
        "unique_timestamps": int(part["time_s"].nunique())
    })

print("\n3. PREDICTION SPREAD AT COMMON EXACT TIMESTAMPS")
counts = (
    d.groupby("time_s")["cell_id"]
    .nunique()
)
shared_times = counts[counts >= 90].index
shared = d[d["time_s"].isin(shared_times)]

spread = (
    shared.groupby("time_s")["predicted_future_soh"]
    .agg(["count", "std", "min", "max"])
)

spread["prediction_range"] = spread["max"] - spread["min"]

print("Timestamps with at least 90 cells:", len(spread))
if not spread.empty:
    print(spread["prediction_range"].describe().to_string())
    print("\nLargest-spread examples:")
    print(
        spread.sort_values("prediction_range", ascending=False)
        .head(10)
        .to_string()
    )

print("\n4. CELL 37 VS CELL 1 NEAR THE END OF CELL 1'S WINDOW")
for t in [1000, 3000, 5000, 7000]:
    nearby = d[d["time_s"].between(t - 1, t + 1)]
    by_cell = nearby.groupby("cell_id").agg(
        time_s=("time_s", "first"),
        predicted_future_soh=("predicted_future_soh", "first")
    )
    print(f"\nAround t={t}s:")
    print(by_cell.loc[by_cell.index.isin([1, 4, 19, 37, 55, 82, 91])])

print("\nAudit complete. No files or metrics were modified.")


1. PER-CELL COVERAGE
         rows  unique_times  first_time_s    last_time_s  prediction_std  prediction_min  prediction_max
cell_id                                                                                                 
1        2691          2690           0.0    7227.598871        0.000007        0.999911        0.999943
2        2691          2690           0.0    7227.598871        0.000007        0.999911        0.999943
3        2691          2690           0.0    7227.598871        0.000007        0.999911        0.999943
4        2691          2690           0.0    7227.598871        0.000007        0.999911        0.999943
5        2691          2690           0.0    7227.598871        0.000007        0.999911        0.999943
6        2691          2690           0.0    7227.598871        0.000007        0.999911        0.999943
7        2691          2690           0.0    7227.598871        0.000007        0.999911        0.999943
8        2691          2690       

In [16]:

# CELL 10F — Audit raw model inputs for faulty vs ordinary cells

import pandas as pd
import numpy as np
from pathlib import Path

# Checkpoint contract: 8 inputs -> future SOH
INPUT_COLS = [
    "cell_voltage_V",
    "cell_current_A",
    "cell_temperature_K",
    "surface_temperature_K",
    "soc_proxy",
    "requested_charge_rate_C",
    "ambient_temperature_C",
    "horizon_s",
]

TARGET = "capacity_retention_soh__future"

cell_path = Path(
    "/content/drive/MyDrive/PINN_EV_Battery/data/v1/cell_timeseries.csv"
)

SCENARIO = "S4_CELL037_BENCH"
CHECK_CELLS = [1, 4, 19, 37, 55, 82, 91]

if not cell_path.exists():
    raise FileNotFoundError(f"Source file not found: {cell_path}")

# Read the CSV header without loading the complete dataset
header = pd.read_csv(cell_path, nrows=0).columns.tolist()

required_columns = [
    c for c in [
        "scenario_instance_id",
        "scenario_id",
        "cell_id",
        "time_s",
        "horizon_s",
        TARGET,
    ] + INPUT_COLS
    if c in header
]

missing_columns = sorted(set(
    [
        "scenario_instance_id",
        "scenario_id",
        "cell_id",
        "time_s",
        TARGET,
    ] + INPUT_COLS
) - set(header))

print("Source:", cell_path)
print("Target:", TARGET)
print("Input columns:", INPUT_COLS)
print("Missing expected columns:", missing_columns)

# Load only the columns available in the CSV
df = pd.read_csv(cell_path, usecols=required_columns)

if "scenario_instance_id" not in df.columns:
    raise KeyError("scenario_instance_id is missing from the source CSV.")

if "cell_id" not in df.columns:
    raise KeyError("cell_id is missing from the source CSV.")

if "time_s" not in df.columns:
    raise KeyError("time_s is missing from the source CSV.")

# Filter the scenario if scenario_instance_id exists
df["cell_id"] = pd.to_numeric(df["cell_id"], errors="coerce")

scenario_df = df[
    df["scenario_instance_id"].astype(str).eq(SCENARIO)
].copy()

if scenario_df.empty:
    print(
        f"\nNo rows found for scenario_instance_id={SCENARIO!r}."
        "\nAvailable scenario examples:"
    )
    print(df["scenario_instance_id"].dropna().unique()[:20])
else:
    print("\nScenario rows:", len(scenario_df))
    print("Unique cells:", scenario_df["cell_id"].nunique())

    # 1. Compare each input feature across selected cells
    available_inputs = [
        c for c in INPUT_COLS if c in scenario_df.columns
    ]

    for feature in available_inputs:
        print(f"\n--- {feature} ---")
        subset = scenario_df[
            scenario_df["cell_id"].isin(CHECK_CELLS)
        ]

        stats = subset.groupby("cell_id")[feature].agg(
            ["count", "nunique", "mean", "std", "min", "max"]
        )
        print(stats.to_string())

    # 2. Compare inputs at exactly shared timestamps
    if available_inputs:
        shared = scenario_df[
            scenario_df["cell_id"].isin(CHECK_CELLS)
        ].copy()

        for feature in available_inputs:
            shared[feature] = pd.to_numeric(
                shared[feature], errors="coerce"
            )

        # Restrict to timestamps represented by multiple checked cells
        cell_counts = shared.groupby("time_s")["cell_id"].nunique()
        common_times = cell_counts[cell_counts >= 2].index
        shared = shared[shared["time_s"].isin(common_times)]

        print("\n--- Cross-cell feature spread at shared timestamps ---")

        for feature in available_inputs:
            spread = shared.groupby("time_s")[feature].agg(
                lambda x: x.max() - x.min()
            )
            spread = spread.replace([np.inf, -np.inf], np.nan).dropna()

            print(
                f"{feature}: "
                f"shared timestamps={len(spread)}, "
                f"median spread={spread.median() if len(spread) else np.nan}, "
                f"max spread={spread.max() if len(spread) else np.nan}"
            )

    # 3. Confirm time ranges, especially for cell 37
    print("\n--- Time range by cell ---")
    print(
        scenario_df[
            scenario_df["cell_id"].isin(CHECK_CELLS)
        ]
        .groupby("cell_id")["time_s"]
        .agg(["count", "nunique", "min", "max"])
        .to_string()
    )


Source: /content/drive/MyDrive/PINN_EV_Battery/data/v1/cell_timeseries.csv
Target: capacity_retention_soh__future
Input columns: ['cell_voltage_V', 'cell_current_A', 'cell_temperature_K', 'surface_temperature_K', 'soc_proxy', 'requested_charge_rate_C', 'ambient_temperature_C', 'horizon_s']
Missing expected columns: ['capacity_retention_soh__future', 'horizon_s']

Scenario rows: 270094
Unique cells: 100

--- cell_voltage_V ---
         count  nunique      mean       std       min  max
cell_id                                                   
1         2691     2471  3.836009  0.237135  3.364495  4.2
4         2691     2471  3.836009  0.237135  3.364495  4.2
19        2691     2471  3.836009  0.237135  3.364495  4.2
37        3685     3384  3.762527  0.322716  2.500000  4.2
55        2691     2471  3.836009  0.237135  3.364495  4.2
82        2691     2471  3.836009  0.237135  3.364495  4.2
91        2691     2471  3.836009  0.237135  3.364495  4.2

--- cell_current_A ---
         count 

In [20]:

# CELL 10G — Audit existing localization ranks for faulty cell 37

import pandas as pd
import numpy as np
from pathlib import Path

pred_path = Path(
    "/content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/"
    "07_localization_predictions.csv"
)

SCENARIO = "S4_CELL037_BENCH"
CHECK_CELLS = [1, 4, 19, 37, 55, 82, 91]

if not pred_path.exists():
    raise FileNotFoundError(f"File not found: {pred_path}")

columns = pd.read_csv(pred_path, nrows=0).columns.tolist()

required = [
    "scenario_instance_id",
    "scenario_id",
    "horizon_s",
    "cell_id",
    "anomaly_score",
    "rank",
    "is_fault_ground_truth",
]

missing = [c for c in required if c not in columns]
if missing:
    raise KeyError(
        f"Missing columns: {missing}\nAvailable: {columns}"
    )

df = pd.read_csv(pred_path)

df["cell_id"] = pd.to_numeric(df["cell_id"], errors="coerce")
df["horizon_s"] = pd.to_numeric(df["horizon_s"], errors="coerce")
df["anomaly_score"] = pd.to_numeric(
    df["anomaly_score"], errors="coerce"
)
df["rank"] = pd.to_numeric(df["rank"], errors="coerce")

scenario = df[
    df["scenario_instance_id"].astype(str).eq(SCENARIO)
].copy()

if scenario.empty:
    print("Scenario not found:", SCENARIO)
    print("Available scenarios:")
    print(df["scenario_instance_id"].dropna().unique()[:30])
else:
    print("Scenario:", SCENARIO)
    print("Rows:", len(scenario))
    print("Unique cells:", scenario["cell_id"].nunique())

    print("\n--- Cell 37 localization result by horizon ---")
    print(
        scenario[scenario["cell_id"].eq(37)][
            ["scenario_instance_id", "horizon_s", "cell_id",
             "anomaly_score", "rank", "is_fault_ground_truth"]
        ].to_string(index=False)
    )

    print("\n--- Selected cells by horizon ---")
    selected = scenario[
        scenario["cell_id"].isin(CHECK_CELLS)
    ].sort_values(["horizon_s", "rank"])

    print(
        selected[
            ["horizon_s", "cell_id", "anomaly_score",
             "rank", "is_fault_ground_truth"]
        ].to_string(index=False)
    )

    print("\n--- Top 10 cells by anomaly rank per horizon ---")
    for horizon, group in scenario.groupby("horizon_s"):
        print(f"\nHorizon: {horizon:g}s")
        print(
            group.sort_values("rank")[
                ["cell_id", "anomaly_score", "rank",
                 "is_fault_ground_truth"]
            ].head(10).to_string(index=False)
        )

    print("\n--- Score variation and ranking health ---")
    for horizon, group in scenario.groupby("horizon_s"):
        scores = group["anomaly_score"].dropna()
        ranks = group["rank"].dropna()

        print(
            f"Horizon={horizon:g}s | "
            f"unique scores={scores.nunique()} | "
            f"score std={scores.std():.12g} | "
            f"unique ranks={ranks.nunique()} | "
            f"cell 37 rank="
            f"{group.loc[group['cell_id'].eq(37), 'rank'].tolist()}"
        )


Scenario: S4_CELL037_BENCH
Rows: 300
Unique cells: 100

--- Cell 37 localization result by horizon ---
scenario_instance_id  horizon_s  cell_id  anomaly_score  rank  is_fault_ground_truth
    S4_CELL037_BENCH         60       37            0.0    37                      1
    S4_CELL037_BENCH        300       37            0.0    37                      1
    S4_CELL037_BENCH        600       37            0.0    37                      1

--- Selected cells by horizon ---
 horizon_s  cell_id  anomaly_score  rank  is_fault_ground_truth
        60        1            0.0     1                      0
        60        4            0.0     4                      0
        60       19            0.0    19                      0
        60       37            0.0    37                      1
        60       55            0.0    55                      0
        60       82            0.0    82                      0
        60       91            0.0    91                      0
       300

In [21]:

# CELL 10H — Locate the raw prediction-level data for score debugging

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")

candidates = [
    ROOT / "results_v1_localization" / "07_localization_predictions.csv",
    ROOT / "results_v1_localization" / "07_localization_scores.csv",
    ROOT / "results_v1_localization" / "07_localization_metrics.csv",
]

for path in candidates:
    print(f"\nFILE: {path}")
    if not path.exists():
        print("STATUS: NOT FOUND")
        continue

    header = pd.read_csv(path, nrows=0).columns.tolist()
    print("COLUMNS:", header)

    if "predicted_future_soh" in header:
        print("STATUS: Contains predicted SOH values")
    elif "deviation_score" in header:
        print("STATUS: Contains prediction-level deviation scores")
    elif "anomaly_score" in header:
        print("STATUS: Contains aggregated anomaly scores")
    else:
        print("STATUS: Does not contain recognized score columns")



FILE: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_localization_predictions.csv
COLUMNS: ['scenario_instance_id', 'scenario_id', 'horizon_s', 'cell_id', 'anomaly_score', 'rank', 'is_fault_ground_truth']
STATUS: Contains aggregated anomaly scores

FILE: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_localization_scores.csv
COLUMNS: ['scenario_instance_id', 'scenario_id', 'cell_id', 'horizon_s', 'anomaly_score', 'predicted_future_soh', 'time_points']
STATUS: Contains predicted SOH values

FILE: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_localization_metrics.csv
COLUMNS: ['scenario_instance_id', 'scenario_id', 'horizon_s', 'n_cells_scored', 'ground_truth_fault_cells', 'observed_fault_cells', 'missing_fault_cells', 'top_k', 'top_k_cells', 'true_positives_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k', 'mean_observed_fault_rank', 'fault_label_coverage', 'ground_truth_used_only_after_scoring']
STATUS: Does not contain recogni

In [22]:

# CELL 10I — Diagnose aggregated SOH predictions and anomaly scores

from pathlib import Path
import pandas as pd
import numpy as np

path = Path(
    "/content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/"
    "07_localization_scores.csv"
)

df = pd.read_csv(path)

required = [
    "scenario_instance_id",
    "scenario_id",
    "cell_id",
    "horizon_s",
    "anomaly_score",
    "predicted_future_soh",
    "time_points",
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}")

for col in [
    "cell_id", "horizon_s", "anomaly_score",
    "predicted_future_soh", "time_points"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Total rows:", len(df))
print("Unique scenarios:", df["scenario_instance_id"].nunique())
print("Unique cells:", df["cell_id"].nunique())
print("Horizons:", sorted(df["horizon_s"].dropna().unique()))

print("\n--- Prediction variation by scenario and horizon ---")

summary = df.groupby(
    ["scenario_instance_id", "horizon_s"]
).agg(
    cells=("cell_id", "nunique"),
    unique_predictions=("predicted_future_soh", "nunique"),
    prediction_std=("predicted_future_soh", "std"),
    prediction_min=("predicted_future_soh", "min"),
    prediction_max=("predicted_future_soh", "max"),
    unique_scores=("anomaly_score", "nunique"),
    score_std=("anomaly_score", "std"),
    nonzero_scores=(
        "anomaly_score",
        lambda s: int((s.abs() > 1e-12).sum())
    ),
).reset_index()

print(summary.to_string(index=False))

print("\n--- Overall anomaly score distribution ---")
print(df["anomaly_score"].describe())
print("Unique anomaly scores:", df["anomaly_score"].nunique())
print("Nonzero anomaly scores:", int(
    (df["anomaly_score"].abs() > 1e-12).sum()
))

print("\n--- Fault scenario: S4_CELL037_BENCH ---")
case = df[
    df["scenario_instance_id"].astype(str).eq("S4_CELL037_BENCH")
]

print(
    case.sort_values(["horizon_s", "cell_id"])[
        ["horizon_s", "cell_id", "predicted_future_soh",
         "anomaly_score", "time_points"]
    ].to_string(index=False)
)


Total rows: 6000
Unique scenarios: 20
Unique cells: 100
Horizons: [np.int64(60), np.int64(300), np.int64(600)]

--- Prediction variation by scenario and horizon ---
    scenario_instance_id  horizon_s  cells  unique_predictions  prediction_std  prediction_min  prediction_max  unique_scores  score_std  nonzero_scores
        S4_CELL037_BENCH         60    100                   2    7.000000e-08        0.999939        0.999939              1        0.0               0
        S4_CELL037_BENCH        300    100                   2    3.040000e-07        0.999939        0.999942              1        0.0               0
        S4_CELL037_BENCH        600    100                   2    3.050000e-07        0.999945        0.999948              1        0.0               0
              S4_CELL082         60    100                   2    7.000000e-08        0.999939        0.999939              1        0.0               0
              S4_CELL082        300    100                   2    3.04

In [23]:

# CELL 10J — Independent signal-based localization baseline
# Ground-truth labels are NOT used to calculate anomaly scores.

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")
SOURCE = ROOT / "data/v1/cell_timeseries.csv"
OUT = ROOT / "results_v1_localization/07_signal_localization_diagnostic.csv"

FEATURES = [
    "cell_voltage_V",
    "cell_current_A",
    "cell_temperature_K",
    "surface_temperature_K",
    "soc_proxy",
    "requested_charge_rate_C",
    "ambient_temperature_C",
]

KEYS = ["scenario_instance_id", "scenario_id", "cell_id"]

if not SOURCE.exists():
    raise FileNotFoundError(f"Source file not found: {SOURCE}")

header = pd.read_csv(SOURCE, nrows=0).columns.tolist()
required = KEYS + ["time_s"] + FEATURES
missing = [c for c in required if c not in header]

if missing:
    raise KeyError(f"Missing source columns: {missing}")

# Accumulate statistics without loading the entire CSV into memory.
partial_results = []
agg_spec = {}

for f in FEATURES:
    agg_spec[f"{f}__count"] = (f, "count")
    agg_spec[f"{f}__sum"] = (f, "sum")
    agg_spec[f"{f}__sumsq"] = (
        f, lambda s: np.square(s).sum()
    )
    agg_spec[f"{f}__min"] = (f, "min")
    agg_spec[f"{f}__max"] = (f, "max")

for chunk in pd.read_csv(
    SOURCE,
    usecols=required,
    chunksize=250_000
):
    chunk["cell_id"] = pd.to_numeric(
        chunk["cell_id"], errors="coerce"
    )

    for f in FEATURES:
        chunk[f] = pd.to_numeric(chunk[f], errors="coerce")

    chunk = chunk.dropna(subset=KEYS)

    if chunk.empty:
        continue

    part = (
        chunk.groupby(KEYS, dropna=False)
        .agg(**agg_spec)
        .reset_index()
    )
    partial_results.append(part)

    del chunk, part

if not partial_results:
    raise ValueError("No usable source rows were found.")

# Combine chunk-level statistics correctly.
partials = pd.concat(partial_results, ignore_index=True)
combine_spec = {}

for f in FEATURES:
    combine_spec[f"{f}__count"] = (f"{f}__count", "sum")
    combine_spec[f"{f}__sum"] = (f"{f}__sum", "sum")
    combine_spec[f"{f}__sumsq"] = (f"{f}__sumsq", "sum")
    combine_spec[f"{f}__min"] = (f"{f}__min", "min")
    combine_spec[f"{f}__max"] = (f"{f}__max", "max")

cell_stats = (
    partials.groupby(KEYS, dropna=False)
    .agg(**combine_spec)
    .reset_index()
)

# Create per-cell summaries for every scenario.
summary_cols = []

for f in FEATURES:
    count = cell_stats[f"{f}__count"].replace(0, np.nan)
    mean = cell_stats[f"{f}__sum"] / count

    variance = (
        cell_stats[f"{f}__sumsq"] / count - mean.pow(2)
    ).clip(lower=0)

    cell_stats[f"{f}__mean"] = mean
    cell_stats[f"{f}__std"] = np.sqrt(variance)

    summary_cols.extend([
        f"{f}__mean",
        f"{f}__std",
        f"{f}__min",
        f"{f}__max",
    ])

# Score each cell by deviations from its peers in the same scenario.
# When MAD is zero, fall back to standard deviation.
results = []

for scenario_id, group in cell_stats.groupby(
    "scenario_instance_id", sort=False
):
    group = group.reset_index(drop=True)
    matrix = group[summary_cols].replace(
        [np.inf, -np.inf], np.nan
    )

    medians = matrix.median(axis=0)
    mad = matrix.sub(medians).abs().median(axis=0)
    std = matrix.std(axis=0, ddof=0)

    scale = (1.4826 * mad).where(
        (1.4826 * mad) > 1e-12, std
    )
    scale = scale.where(scale > 1e-12, np.nan)

    z = (
        matrix.sub(medians).abs()
        .div(scale, axis=1)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # Average the largest five feature-summary deviations.
    # This is an exploratory score, not a calibrated probability.
    values = z.to_numpy()
    n_top = min(5, values.shape[1])
    top_values = np.sort(values, axis=1)[:, -n_top:]
    scores = top_values.mean(axis=1)

    output = group[KEYS].copy()
    output["signal_anomaly_score"] = scores
    output["signal_rank"] = (
        output["signal_anomaly_score"]
        .rank(method="min", ascending=False)
        .astype(int)
    )

    results.append(output)

result = pd.concat(results, ignore_index=True)
result = result.sort_values(
    ["scenario_instance_id", "signal_rank", "cell_id"]
)

OUT.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT, index=False)

print("Saved diagnostic:", OUT)
print("Rows:", len(result))
print("Scenarios:", result["scenario_instance_id"].nunique())
print("Cells per scenario (min/max):",
      result.groupby("scenario_instance_id")["cell_id"].nunique().min(),
      result.groupby("scenario_instance_id")["cell_id"].nunique().max())

# Inspect a known scenario. The cell ID is not used in the score.
example = result[
    result["scenario_instance_id"].astype(str).eq(
        "S4_CELL037_BENCH"
    )
].sort_values("signal_rank")

print("\nTop 10 cells for S4_CELL037_BENCH:")
print(example.head(10).to_string(index=False))

print("\nCell 37 diagnostic rank:")
print(
    example[example["cell_id"].eq(37)].to_string(index=False)
)


Saved diagnostic: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_signal_localization_diagnostic.csv
Rows: 2800
Scenarios: 28
Cells per scenario (min/max): 100 100

Top 10 cells for S4_CELL037_BENCH:
scenario_instance_id scenario_id  cell_id  signal_anomaly_score  signal_rank
    S4_CELL037_BENCH          S4       37          1.005038e+01            1
    S4_CELL037_BENCH          S4       30          2.922397e-10            2
    S4_CELL037_BENCH          S4        1          0.000000e+00            3
    S4_CELL037_BENCH          S4        2          0.000000e+00            3
    S4_CELL037_BENCH          S4        3          0.000000e+00            3
    S4_CELL037_BENCH          S4        4          0.000000e+00            3
    S4_CELL037_BENCH          S4        5          0.000000e+00            3
    S4_CELL037_BENCH          S4        6          0.000000e+00            3
    S4_CELL037_BENCH          S4        7          0.000000e+00            3
    S4_CELL0

In [26]:

# CELL 10K — Evaluate signal-based localization using authoritative metadata

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")

SCORES_PATH = (
    ROOT / "results_v1_localization"
    / "07_signal_localization_diagnostic.csv"
)
META_PATH = ROOT / "data/v1/scenario_instances.csv"

OUT = (
    ROOT / "results_v1_localization"
    / "07_signal_localization_metrics.csv"
)

scores = pd.read_csv(SCORES_PATH)
metadata = pd.read_csv(META_PATH)

required_scores = [
    "scenario_instance_id", "scenario_id",
    "cell_id", "signal_anomaly_score", "signal_rank"
]
required_meta = [
    "scenario_instance_id", "scenario_id", "fault_cells_json"
]

for col in required_scores:
    if col not in scores.columns:
        raise KeyError(f"Missing score column: {col}")

for col in required_meta:
    if col not in metadata.columns:
        raise KeyError(f"Missing metadata column: {col}")

scores["cell_id"] = pd.to_numeric(
    scores["cell_id"], errors="raise"
).astype(int)
scores["signal_anomaly_score"] = pd.to_numeric(
    scores["signal_anomaly_score"], errors="raise"
)

if metadata["scenario_instance_id"].duplicated().any():
    raise ValueError(
        "Metadata contains duplicate scenario_instance_id values. "
        "Resolve duplicates before evaluation."
    )

def parse_fault_cells(value):
    if pd.isna(value) or str(value).strip() in ("", "[]", "null"):
        return set()

    parsed = json.loads(value) if isinstance(value, str) else value

    if parsed is None:
        return set()

    if not isinstance(parsed, list):
        raise ValueError(
            f"Unexpected fault_cells_json format: {value!r}"
        )

    return {int(cell) for cell in parsed}

metadata["fault_cells"] = metadata["fault_cells_json"].apply(
    parse_fault_cells
)

# Verify score/metadata scenario coverage.
score_scenarios = set(scores["scenario_instance_id"].astype(str))
meta_scenarios = set(metadata["scenario_instance_id"].astype(str))

if score_scenarios - meta_scenarios:
    raise ValueError(
        "Scenarios missing from metadata: "
        f"{sorted(score_scenarios - meta_scenarios)}"
    )

results = []

for scenario_id, group in scores.groupby(
    "scenario_instance_id", sort=False
):
    meta = metadata[
        metadata["scenario_instance_id"].astype(str).eq(str(scenario_id))
    ]

    if len(meta) != 1:
        raise ValueError(
            f"Expected one metadata row for {scenario_id}; found {len(meta)}"
        )

    truth = meta.iloc[0]["fault_cells"]
    group = group.sort_values(
        ["signal_anomaly_score", "cell_id"],
        ascending=[False, True]
    ).reset_index(drop=True)

    cells = set(group["cell_id"])
    missing_faults = truth - cells

    if missing_faults:
        raise ValueError(
            f"{scenario_id}: ground-truth fault cells missing "
            f"from scores: {sorted(missing_faults)}"
        )

    k = len(truth)
    top_cells = set(group.head(k)["cell_id"]) if k else set()
    tp = len(top_cells & truth)

    precision = tp / k if k else np.nan
    recall = tp / len(truth) if truth else np.nan
    f1 = (
        2 * precision * recall / (precision + recall)
        if truth and precision + recall > 0
        else (0.0 if truth else np.nan)
    )

    ranks = {
        int(row.cell_id): int(index + 1)
        for index, row in group.iterrows()
    }
    fault_ranks = [ranks[cell] for cell in sorted(truth)]

    results.append({
        "scenario_instance_id": scenario_id,
        "scenario_id": meta.iloc[0]["scenario_id"],
        "cells_scored": group["cell_id"].nunique(),
        "ground_truth_fault_cells": json.dumps(sorted(truth)),
        "fault_count": len(truth),
        "top_k": k,
        "top_k_cells": json.dumps(sorted(top_cells)),
        "true_positives_at_k": tp,
        "precision_at_k": precision,
        "recall_at_k": recall,
        "f1_at_k": f1,
        "fault_ranks": json.dumps(fault_ranks),
        "mean_fault_rank": (
            float(np.mean(fault_ranks)) if fault_ranks else np.nan
        ),
        "all_faults_present": not missing_faults,
        "score_ties": bool(
            group["signal_anomaly_score"].duplicated().any()
        ),
    })

metrics = pd.DataFrame(results)
metrics.to_csv(OUT, index=False)

faulty = metrics[metrics["fault_count"] > 0]
healthy = metrics[metrics["fault_count"] == 0]

print("Saved:", OUT)
print("Scenarios evaluated:", len(metrics))
print("Scenarios with 100 cells:",
      int((metrics["cells_scored"] == 100).sum()))

print("\n--- Fault scenarios ---")
print(
    faulty[
        ["scenario_instance_id", "ground_truth_fault_cells",
         "top_k_cells", "fault_ranks", "recall_at_k"]
    ].to_string(index=False)
)

print("\n--- Evaluation summary ---")
print("Fault scenarios:", len(faulty))
print("Healthy scenarios:", len(healthy))

if not faulty.empty:
    print("Mean Recall@K:", faulty["recall_at_k"].mean())
    print("Mean F1@K:", faulty["f1_at_k"].mean())
    print("Mean fault rank:", faulty["mean_fault_rank"].mean())
    print(
        "Fault scenarios with all faults in top K:",
        int((faulty["recall_at_k"] == 1.0).sum()),
        "/", len(faulty)
    )

print(
    "\nNote: Healthy scenarios have no top-K selection because K=0. "
    "False alarms require a separately defined decision threshold."
)


Saved: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_signal_localization_metrics.csv
Scenarios evaluated: 28
Scenarios with 100 cells: 28

--- Fault scenarios ---
    scenario_instance_id ground_truth_fault_cells top_k_cells fault_ranks  recall_at_k
              S4_CELL004                      [4]         [4]         [1]          1.0
              S4_CELL019                     [19]        [19]         [1]          1.0
        S4_CELL037_BENCH                     [37]        [37]         [1]          1.0
              S4_CELL055                     [55]        [55]         [1]          1.0
              S4_CELL082                     [82]        [82]         [1]          1.0
              S5_CELL004                      [4]         [4]         [1]          1.0
          S5_CELL004_082                  [4, 82]     [4, 82]      [1, 2]          1.0
        S5_CELL037_BENCH                     [37]        [37]         [1]          1.0
              S5_CELL073          

In [25]:

# CELL 10K-A — Locate scenario metadata containing fault labels

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")

matches = []

for path in ROOT.rglob("*.csv"):
    # Skip generated outputs to avoid unnecessary scanning
    if "results_v1_localization" in str(path):
        continue

    try:
        header = pd.read_csv(path, nrows=0).columns.tolist()

        if "fault_cells_json" in header:
            matches.append(path)
            print("\nFOUND FAULT METADATA:")
            print(path)
            print("Columns:", header)

    except Exception as exc:
        print(f"Skipped {path.name}: {exc}")

if not matches:
    print(
        "\nNo CSV with fault_cells_json found."
        "\nNext, we need to inspect the scenario metadata source "
        "used to generate the original localization labels."
    )
else:
    print(f"\nTotal matching files: {len(matches)}")



FOUND FAULT METADATA:
/content/drive/MyDrive/PINN_EV_Battery/data/v1/scenario_instances.csv
Columns: ['scenario_instance_id', 'scenario_id', 'split', 'pack_topology', 'cell_count', 'fault_cells_json', 'fault_count', 'fault_location_is_hidden_to_model', 'cell37_is_benchmark', 'healthy_reference_required', 'degradation_ground_truth_source', 'degradation_parameter_multipliers_used', 'threshold_status', 'localization_status', 'base_scenario_instance_id', 'candidate_charge_rate_C', 'winner_preselected']

FOUND FAULT METADATA:
/content/drive/MyDrive/PINN_EV_Battery/data/v1/s6_intervention_summary.csv
Columns: ['scenario_instance_id', 'scenario_id', 'base_scenario_instance_id', 'candidate_charge_rate_C', 'fault_cells_json', 'cells_scanned', 'winner_preselected', 'max_pack_voltage_V', 'min_pack_voltage_V', 'max_abs_current_A']

Total matching files: 2


In [27]:

# CELL 10L — Threshold-based fault detection and false-alarm evaluation

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")

SCORES_PATH = (
    ROOT / "results_v1_localization"
    / "07_signal_localization_diagnostic.csv"
)
META_PATH = ROOT / "data/v1/scenario_instances.csv"

OUT_CELL = (
    ROOT / "results_v1_localization"
    / "07_threshold_cell_predictions.csv"
)
OUT_SCENARIO = (
    ROOT / "results_v1_localization"
    / "07_threshold_evaluation.csv"
)

scores = pd.read_csv(SCORES_PATH)
meta = pd.read_csv(META_PATH)

scores["cell_id"] = pd.to_numeric(
    scores["cell_id"], errors="raise"
).astype(int)
scores["signal_anomaly_score"] = pd.to_numeric(
    scores["signal_anomaly_score"], errors="raise"
)

def parse_faults(value):
    if pd.isna(value) or str(value).strip() in ("", "[]", "null"):
        return set()
    value = json.loads(value) if isinstance(value, str) else value
    if value is None:
        return set()
    if not isinstance(value, list):
        raise ValueError(f"Unexpected fault label format: {value!r}")
    return {int(x) for x in value}

if meta["scenario_instance_id"].duplicated().any():
    raise ValueError("Duplicate scenario metadata IDs found.")

meta["fault_cells"] = meta["fault_cells_json"].apply(parse_faults)

# Match every score scenario to its authoritative metadata.
meta_index = meta.set_index("scenario_instance_id")
score_ids = set(scores["scenario_instance_id"].astype(str))
meta_ids = set(meta["scenario_instance_id"].astype(str))

if score_ids - meta_ids:
    raise ValueError(
        f"Missing metadata for: {sorted(score_ids - meta_ids)}"
    )

scores["scenario_instance_id"] = scores[
    "scenario_instance_id"
].astype(str)
meta["scenario_instance_id"] = meta[
    "scenario_instance_id"
].astype(str)

meta_index = meta.set_index("scenario_instance_id")

scores["fault_cells"] = scores["scenario_instance_id"].map(
    meta_index["fault_cells"]
)
scores["is_healthy_scenario"] = scores["fault_cells"].apply(
    lambda x: len(x) == 0
)
scores["is_fault_ground_truth"] = scores.apply(
    lambda row: int(row["cell_id"] in row["fault_cells"]),
    axis=1
)

# Calibrate a preliminary threshold using only cells in healthy scenarios.
healthy_scores = scores.loc[
    scores["is_healthy_scenario"], "signal_anomaly_score"
].dropna()

if healthy_scores.empty:
    raise ValueError("No healthy scenarios available for threshold calibration.")

threshold = float(healthy_scores.max())

# Strictly greater than threshold means flagged.
scores["predicted_defective"] = (
    scores["signal_anomaly_score"] > threshold
).astype(int)

# Overall cell-level confusion counts.
tp = int((
    (scores["predicted_defective"] == 1)
    & (scores["is_fault_ground_truth"] == 1)
).sum())
fp = int((
    (scores["predicted_defective"] == 1)
    & (scores["is_fault_ground_truth"] == 0)
).sum())
fn = int((
    (scores["predicted_defective"] == 0)
    & (scores["is_fault_ground_truth"] == 1)
).sum())
tn = int((
    (scores["predicted_defective"] == 0)
    & (scores["is_fault_ground_truth"] == 0)
).sum())

precision = tp / (tp + fp) if tp + fp else np.nan
recall = tp / (tp + fn) if tp + fn else np.nan
specificity = tn / (tn + fp) if tn + fp else np.nan
f1 = (
    2 * precision * recall / (precision + recall)
    if pd.notna(precision) and pd.notna(recall)
    and precision + recall > 0 else np.nan
)

print("Healthy calibration scenarios:",
      int(meta["fault_cells"].apply(len).eq(0).sum()))
print("Threshold = maximum cell score across healthy scenarios:",
      f"{threshold:.12g}")
print("\n--- Cell-level results ---")
print("TP:", tp, "| FP:", fp, "| FN:", fn, "| TN:", tn)
print("Precision:", precision)
print("Recall / sensitivity:", recall)
print("Specificity:", specificity)
print("F1:", f1)

# Scenario-level evaluation.
scenario_rows = []

for scenario_id, group in scores.groupby("scenario_instance_id"):
    truth = group["fault_cells"].iloc[0]
    actual = set(truth)
    predicted = set(
        group.loc[group["predicted_defective"] == 1, "cell_id"]
    )

    scenario_rows.append({
        "scenario_instance_id": scenario_id,
        "n_cells": group["cell_id"].nunique(),
        "ground_truth_fault_cells": json.dumps(sorted(actual)),
        "predicted_defective_cells": json.dumps(sorted(predicted)),
        "true_positives": len(actual & predicted),
        "false_positives": len(predicted - actual),
        "false_negatives": len(actual - predicted),
        "healthy_scenario": len(actual) == 0,
        "healthy_false_alarm_count": (
            len(predicted) if len(actual) == 0 else np.nan
        ),
        "all_faults_detected": (
            actual.issubset(predicted) if actual else np.nan
        ),
    })

scenario_results = pd.DataFrame(scenario_rows)

scores.to_csv(OUT_CELL, index=False)
scenario_results.to_csv(OUT_SCENARIO, index=False)

print("\nSaved cell predictions:", OUT_CELL)
print("Saved scenario evaluation:", OUT_SCENARIO)

print("\n--- Healthy scenarios ---")
print(
    scenario_results[scenario_results["healthy_scenario"]][
        ["scenario_instance_id", "predicted_defective_cells",
         "healthy_false_alarm_count"]
    ].to_string(index=False)
)

print("\n--- Fault scenarios ---")
print(
    scenario_results[~scenario_results["healthy_scenario"]][
        ["scenario_instance_id", "ground_truth_fault_cells",
         "predicted_defective_cells", "false_positives",
         "false_negatives", "all_faults_detected"]
    ].to_string(index=False)
)

print(
    "\nCAUTION: This is a preliminary threshold calibrated and evaluated "
    "on the available scenarios. The same healthy scenarios are used "
    "to choose and assess the threshold, so independent validation is "
    "still required."
)


Healthy calibration scenarios: 3
Threshold = maximum cell score across healthy scenarios: 0

--- Cell-level results ---
TP: 31 | FP: 27 | FN: 0 | TN: 2742
Precision: 0.5344827586206896
Recall / sensitivity: 1.0
Specificity: 0.9902491874322861
F1: 0.6966292134831461

Saved cell predictions: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_threshold_cell_predictions.csv
Saved scenario evaluation: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_threshold_evaluation.csv

--- Healthy scenarios ---
scenario_instance_id predicted_defective_cells  healthy_false_alarm_count
          S1_HEALTHY                        []                        0.0
          S2_HEALTHY                        []                        0.0
          S3_HEALTHY                        []                        0.0

--- Fault scenarios ---
    scenario_instance_id ground_truth_fault_cells predicted_defective_cells  false_positives  false_negatives all_faults_detected
              S4

In [28]:

# CELL 10M — Diagnose false positives and score separation

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery/results_v1_localization")

scores_path = ROOT / "07_signal_localization_diagnostic.csv"
meta_path = (
    Path("/content/drive/MyDrive/PINN_EV_Battery")
    / "data/v1/scenario_instances.csv"
)

scores = pd.read_csv(scores_path)
meta = pd.read_csv(meta_path)

def parse_faults(value):
    if pd.isna(value) or str(value).strip() in ("", "[]", "null"):
        return set()
    parsed = json.loads(value) if isinstance(value, str) else value
    return set() if parsed is None else {int(x) for x in parsed}

meta["fault_cells"] = meta["fault_cells_json"].apply(parse_faults)
meta["scenario_instance_id"] = meta["scenario_instance_id"].astype(str)
scores["scenario_instance_id"] = scores["scenario_instance_id"].astype(str)

truth_map = meta.set_index("scenario_instance_id")["fault_cells"].to_dict()

scores["cell_id"] = pd.to_numeric(scores["cell_id"]).astype(int)
scores["signal_anomaly_score"] = pd.to_numeric(
    scores["signal_anomaly_score"]
)
scores["is_fault"] = scores.apply(
    lambda r: int(
        r["cell_id"] in truth_map[r["scenario_instance_id"]]
    ), axis=1
)

print("--- Overall score distribution ---")
print(scores["signal_anomaly_score"].describe(percentiles=[
    .25, .50, .75, .90, .95, .99
]).to_string())

print("\nUnique score count:", scores["signal_anomaly_score"].nunique())
print("Exact zeros:", int((scores["signal_anomaly_score"] == 0).sum()))
print("Tiny positive scores (0, 1e-8]:", int((
    (scores["signal_anomaly_score"] > 0)
    & (scores["signal_anomaly_score"] <= 1e-8)
).sum()))

print("\n--- Fault vs healthy-cell score distributions ---")
for label, group in scores.groupby("is_fault"):
    name = "GROUND-TRUTH FAULTY CELLS" if label else "OTHER CELLS"
    print(f"\n{name}")
    print(group["signal_anomaly_score"].describe(
        percentiles=[.5, .9, .95, .99]
    ).to_string())

print("\n--- Top 3 cells per fault scenario ---")
for scenario, group in scores.groupby("scenario_instance_id"):
    truth = truth_map[scenario]
    if not truth:
        continue

    group = group.sort_values(
        "signal_anomaly_score", ascending=False
    ).reset_index(drop=True)

    print(f"\n{scenario} | true faulty cells: {sorted(truth)}")
    print(group.head(3)[
        ["cell_id", "signal_anomaly_score", "is_fault"]
    ].to_string(index=False))

print("\n--- Score overlap check ---")
fault_scores = scores.loc[
    scores["is_fault"] == 1, "signal_anomaly_score"
]
other_scores = scores.loc[
    scores["is_fault"] == 0, "signal_anomaly_score"
]

if not fault_scores.empty and not other_scores.empty:
    print("Minimum faulty-cell score:", fault_scores.min())
    print("Maximum other-cell score:", other_scores.max())
    print(
        "Do score ranges overlap?",
        bool(fault_scores.min() <= other_scores.max())
    )


--- Overall score distribution ---
count    2800.000000
mean        0.100999
std         0.947426
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
90%         0.000000
95%         0.000000
99%         7.142857
max        10.050378

Unique score count: 46
Exact zeros: 2742
Tiny positive scores (0, 1e-8]: 23

--- Fault vs healthy-cell score distributions ---

OTHER CELLS
count    2769.000000
mean        0.002212
std         0.058706
min         0.000000
50%         0.000000
90%         0.000000
95%         0.000000
99%         0.000000
max         1.655212

GROUND-TRUTH FAULTY CELLS
count    31.000000
mean      8.924886
std       1.439623
min       7.142857
50%      10.050378
90%      10.050378
95%      10.050378
99%      10.050378
max      10.050378

--- Top 3 cells per fault scenario ---

S4_CELL004 | true faulty cells: [4]
 cell_id  signal_anomaly_score  is_fault
       4          1.005038e+01         1
      70          1.354084e-11         0
      

In [29]:

# CELL 10N — Evaluate a provisional signal anomaly threshold

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")
SCORE_FILE = ROOT / "results_v1_localization/07_signal_localization_diagnostic.csv"
META_FILE = ROOT / "data/v1/scenario_instances.csv"

OUT_CELLS = ROOT / "results_v1_localization/07_threshold_v2_cell_predictions.csv"
OUT_SCENARIOS = ROOT / "results_v1_localization/07_threshold_v2_metrics.csv"

# Provisional threshold based on the observed score separation.
# Must be validated on held-out scenarios before deployment.
THRESHOLD = 4.39903446

scores = pd.read_csv(SCORE_FILE)
meta = pd.read_csv(META_FILE)

def parse_faults(value):
    if pd.isna(value) or str(value).strip() in ("", "[]", "null"):
        return set()
    parsed = json.loads(value) if isinstance(value, str) else value
    return set() if parsed is None else {int(x) for x in parsed}

if meta["scenario_instance_id"].duplicated().any():
    raise ValueError("Duplicate scenario IDs in metadata.")

meta["scenario_instance_id"] = meta["scenario_instance_id"].astype(str)
scores["scenario_instance_id"] = scores["scenario_instance_id"].astype(str)
meta["fault_cells"] = meta["fault_cells_json"].apply(parse_faults)

truth_map = meta.set_index("scenario_instance_id")["fault_cells"].to_dict()

scores["cell_id"] = pd.to_numeric(scores["cell_id"]).astype(int)
scores["signal_anomaly_score"] = pd.to_numeric(
    scores["signal_anomaly_score"], errors="raise"
)

missing = set(scores["scenario_instance_id"]) - set(truth_map)
if missing:
    raise ValueError(f"Missing metadata for scenarios: {sorted(missing)}")

scores["is_fault_ground_truth"] = scores.apply(
    lambda r: int(
        r["cell_id"] in truth_map[r["scenario_instance_id"]]
    ),
    axis=1,
)

scores["predicted_defective"] = (
    scores["signal_anomaly_score"] >= THRESHOLD
).astype(int)

tp = int(((scores["predicted_defective"] == 1) &
          (scores["is_fault_ground_truth"] == 1)).sum())
fp = int(((scores["predicted_defective"] == 1) &
          (scores["is_fault_ground_truth"] == 0)).sum())
fn = int(((scores["predicted_defective"] == 0) &
          (scores["is_fault_ground_truth"] == 1)).sum())
tn = int(((scores["predicted_defective"] == 0) &
          (scores["is_fault_ground_truth"] == 0)).sum())

precision = tp / (tp + fp) if tp + fp else np.nan
recall = tp / (tp + fn) if tp + fn else np.nan
specificity = tn / (tn + fp) if tn + fp else np.nan
f1 = (
    2 * precision * recall / (precision + recall)
    if pd.notna(precision) and pd.notna(recall)
    and precision + recall > 0 else np.nan
)

scenario_rows = []

for scenario, group in scores.groupby("scenario_instance_id"):
    truth = truth_map[scenario]
    predicted = set(
        group.loc[group["predicted_defective"] == 1, "cell_id"]
    )

    scenario_rows.append({
        "scenario_instance_id": scenario,
        "ground_truth_fault_cells": json.dumps(sorted(truth)),
        "predicted_defective_cells": json.dumps(sorted(predicted)),
        "fault_count": len(truth),
        "true_positives": len(truth & predicted),
        "false_positives": len(predicted - truth),
        "false_negatives": len(truth - predicted),
        "healthy_scenario": len(truth) == 0,
        "all_faults_detected": (
            truth.issubset(predicted) if truth else np.nan
        ),
    })

scenario_metrics = pd.DataFrame(scenario_rows)

scores.to_csv(OUT_CELLS, index=False)
scenario_metrics.to_csv(OUT_SCENARIOS, index=False)

print("Threshold:", THRESHOLD)
print("\n--- Cell-level metrics ---")
print("TP:", tp, "FP:", fp, "FN:", fn, "TN:", tn)
print("Precision:", precision)
print("Recall:", recall)
print("Specificity:", specificity)
print("F1:", f1)

print("\n--- Scenario-level results ---")
print("Scenarios:", len(scenario_metrics))
print(
    "Fault scenarios with every fault detected:",
    int(scenario_metrics.loc[
        scenario_metrics["fault_count"] > 0,
        "all_faults_detected"
    ].sum()),
    "/",
    int((scenario_metrics["fault_count"] > 0).sum())
)

print("\n--- Healthy scenarios ---")
print(
    scenario_metrics[scenario_metrics["healthy_scenario"]][
        ["scenario_instance_id", "predicted_defective_cells"]
    ].to_string(index=False)
)

print("\nSaved:", OUT_CELLS)
print("Saved:", OUT_SCENARIOS)
print(
    "\nIMPORTANT: The threshold was selected using the observed score "
    "ranges. These metrics are diagnostic, not independent test results."
)


Threshold: 4.39903446

--- Cell-level metrics ---
TP: 31 FP: 0 FN: 0 TN: 2769
Precision: 1.0
Recall: 1.0
Specificity: 1.0
F1: 1.0

--- Scenario-level results ---
Scenarios: 28
Fault scenarios with every fault detected: 25 / 25

--- Healthy scenarios ---
scenario_instance_id predicted_defective_cells
          S1_HEALTHY                        []
          S2_HEALTHY                        []
          S3_HEALTHY                        []

Saved: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_threshold_v2_cell_predictions.csv
Saved: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_threshold_v2_metrics.csv

IMPORTANT: The threshold was selected using the observed score ranges. These metrics are diagnostic, not independent test results.


In [30]:

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")
META_PATH = ROOT / "data/v1/scenario_instances.csv"
SCORES_PATH = ROOT / "results_v1_localization/07_signal_localization_diagnostic.csv"

meta = pd.read_csv(META_PATH)
scores = pd.read_csv(SCORES_PATH)

print("=== DATA SPLIT DISTRIBUTION ===")
if "split" in meta.columns:
    print(meta["split"].fillna("MISSING").value_counts(dropna=False))
else:
    print("No split column found.")

print("\n=== SCENARIO FAMILY INFORMATION ===")
cols = [
    c for c in [
        "scenario_instance_id",
        "scenario_id",
        "split",
        "base_scenario_instance_id",
        "fault_cells_json",
        "fault_count",
        "cell37_is_benchmark"
    ]
    if c in meta.columns
]
print(meta[cols].sort_values("scenario_instance_id").to_string(index=False))

print("\n=== BASE-SCENARIO GROUPING ===")
if "base_scenario_instance_id" in meta.columns:
    meta["base_group"] = (
        meta["base_scenario_instance_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )
    missing_base = meta["base_group"].isin(["", "nan", "None"])
    meta.loc[missing_base, "base_group"] = (
        meta.loc[missing_base, "scenario_instance_id"].astype(str)
    )
    print("Unique base groups:", meta["base_group"].nunique())
    print(
        meta.groupby("base_group")
        .agg(
            scenario_count=("scenario_instance_id", "count"),
            scenarios=("scenario_instance_id",
                       lambda x: ", ".join(x.astype(str))),
            splits=("split", lambda x: ", ".join(
                sorted(set(x.dropna().astype(str)))
            ) if "split" in meta.columns else "not available")
        )
        .to_string()
    )
else:
    print("No base_scenario_instance_id column found.")

print("\n=== SCORE / METADATA ALIGNMENT ===")
print("Metadata scenarios:", meta["scenario_instance_id"].nunique())
print("Scored scenarios:", scores["scenario_instance_id"].nunique())
print(
    "Scored scenarios missing metadata:",
    len(set(scores["scenario_instance_id"]) -
        set(meta["scenario_instance_id"]))
)
print(
    "Metadata scenarios missing scores:",
    len(set(meta["scenario_instance_id"]) -
        set(scores["scenario_instance_id"]))
)

print("\n=== SCORE DATASET SUMMARY ===")
print("Score rows:", len(scores))
print("Cells per scenario (min/max):",
      scores.groupby("scenario_instance_id")["cell_id"].nunique().min(),
      scores.groupby("scenario_instance_id")["cell_id"].nunique().max())

print("\nAudit complete. Do not select a final threshold from test labels.")


=== DATA SPLIT DISTRIBUTION ===
split
test     20
train     5
val       3
Name: count, dtype: int64

=== SCENARIO FAMILY INFORMATION ===
    scenario_instance_id scenario_id split base_scenario_instance_id fault_cells_json  fault_count  cell37_is_benchmark
              S1_HEALTHY          S1 train                       NaN               []            0                False
              S2_HEALTHY          S2 train                       NaN               []            0                False
              S3_HEALTHY          S3   val                       NaN               []            0                False
              S4_CELL004          S4 train                       NaN              [4]            1                False
              S4_CELL019          S4 train                       NaN             [19]            1                False
        S4_CELL037_BENCH          S4  test                       NaN             [37]            1                 True
              S4_CELL05

In [31]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")
RESULTS = ROOT / "results_v1_localization"

SCORES_PATH = RESULTS / "07_signal_localization_diagnostic.csv"
META_PATH = ROOT / "data/v1/scenario_instances.csv"

scores = pd.read_csv(SCORES_PATH)
meta = pd.read_csv(META_PATH)

# Use labels only to calibrate on training data and evaluate each split.
meta["fault_cells"] = meta["fault_cells_json"].apply(
    lambda x: [int(c) for c in json.loads(x)]
    if pd.notna(x) else []
)

df = scores.merge(
    meta[["scenario_instance_id", "split", "fault_cells"]],
    on="scenario_instance_id",
    how="left",
    validate="many_to_one"
)

assert df["split"].notna().all(), "Some scored scenarios lack split metadata."
assert df["signal_anomaly_score"].notna().all(), "Missing anomaly scores."

df["is_fault"] = df.apply(
    lambda r: int(int(r["cell_id"]) in r["fault_cells"]), axis=1
)

# -------------------------------------------------------
# 1. Choose threshold using TRAIN ONLY
# -------------------------------------------------------
train = df[df["split"] == "train"].copy()

train_fault_scores = train.loc[train["is_fault"] == 1,
                               "signal_anomaly_score"]
train_other_scores = train.loc[train["is_fault"] == 0,
                               "signal_anomaly_score"]

if train_fault_scores.empty or train_other_scores.empty:
    raise ValueError("Training split must contain faulty and non-faulty cells.")

max_train_other = float(train_other_scores.max())
min_train_fault = float(train_fault_scores.min())

print("=== TRAIN-ONLY THRESHOLD CALIBRATION ===")
print("Training scenarios:", train["scenario_instance_id"].nunique())
print("Maximum non-fault training score:", max_train_other)
print("Minimum faulty training score:", min_train_fault)

if max_train_other >= min_train_fault:
    raise ValueError(
        "Training scores overlap. A perfectly separating midpoint "
        "threshold cannot be selected from this training split."
    )

THRESHOLD = (max_train_other + min_train_fault) / 2
print("Selected threshold:", THRESHOLD)

# -------------------------------------------------------
# 2. Apply fixed threshold to all splits
# -------------------------------------------------------
df["predicted_defective"] = (
    df["signal_anomaly_score"] > THRESHOLD
).astype(int)

def calculate_metrics(part):
    y = part["is_fault"].to_numpy(dtype=int)
    p = part["predicted_defective"].to_numpy(dtype=int)

    tp = int(((y == 1) & (p == 1)).sum())
    fp = int(((y == 0) & (p == 1)).sum())
    fn = int(((y == 1) & (p == 0)).sum())
    tn = int(((y == 0) & (p == 0)).sum())

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    specificity = tn / (tn + fp) if tn + fp else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    return {
        "split": part["split"].iloc[0],
        "scenarios": part["scenario_instance_id"].nunique(),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1
    }

metric_rows = []
for split_name in ["train", "val", "test"]:
    part = df[df["split"] == split_name]
    if not part.empty:
        metric_rows.append(calculate_metrics(part))

metrics = pd.DataFrame(metric_rows)

# -------------------------------------------------------
# 3. Scenario-level fault detection and healthy false alarms
# -------------------------------------------------------
scenario_rows = []

for scenario_id, group in df.groupby("scenario_instance_id"):
    actual = sorted(set(group["fault_cells"].iloc[0]))
    predicted = sorted(
        group.loc[group["predicted_defective"] == 1, "cell_id"]
        .astype(int).tolist()
    )

    detected = sorted(set(actual) & set(predicted))
    missed = sorted(set(actual) - set(predicted))
    false_alarms = sorted(set(predicted) - set(actual))

    scenario_rows.append({
        "scenario_instance_id": scenario_id,
        "split": group["split"].iloc[0],
        "actual_fault_cells": json.dumps(actual),
        "predicted_defective_cells": json.dumps(predicted),
        "detected_fault_cells": json.dumps(detected),
        "missed_fault_cells": json.dumps(missed),
        "false_alarm_cells": json.dumps(false_alarms),
        "all_faults_detected": len(missed) == 0,
        "healthy_scenario": len(actual) == 0,
        "healthy_false_alarm_free": len(actual) != 0 or len(predicted) == 0
    })

scenario_results = pd.DataFrame(scenario_rows)

print("\n=== CELL-LEVEL METRICS BY SPLIT ===")
print(metrics.to_string(index=False))

print("\n=== SCENARIO-LEVEL RESULTS ===")
for split_name in ["train", "val", "test"]:
    part = scenario_results[scenario_results["split"] == split_name]
    fault_part = part[~part["healthy_scenario"]]
    healthy_part = part[part["healthy_scenario"]]

    if len(fault_part):
        print(
            f"{split_name}: all faults detected in "
            f"{int(fault_part['all_faults_detected'].sum())}/"
            f"{len(fault_part)} fault scenarios"
        )

    if len(healthy_part):
        print(
            f"{split_name}: healthy scenarios without false alarms "
            f"{int(healthy_part['healthy_false_alarm_free'].sum())}/"
            f"{len(healthy_part)}"
        )

print("\n=== TEST SCENARIO DETAILS ===")
print(
    scenario_results[scenario_results["split"] == "test"][
        [
            "scenario_instance_id",
            "actual_fault_cells",
            "predicted_defective_cells",
            "missed_fault_cells",
            "false_alarm_cells"
        ]
    ].to_string(index=False)
)

# -------------------------------------------------------
# 4. Save auditable results
# -------------------------------------------------------
pred_path = RESULTS / "07_threshold_v3_predictions.csv"
metrics_path = RESULTS / "07_threshold_v3_metrics_by_split.csv"
scenario_path = RESULTS / "07_threshold_v3_scenario_results.csv"
threshold_path = RESULTS / "07_threshold_v3_calibration.json"

df.drop(columns=["fault_cells"]).to_csv(pred_path, index=False)
metrics.to_csv(metrics_path, index=False)
scenario_results.to_csv(scenario_path, index=False)

with open(threshold_path, "w") as f:
    json.dump({
        "threshold": THRESHOLD,
        "selection_method": "midpoint between maximum non-fault and minimum faulty training scores",
        "calibration_split": "train",
        "validation_used_for_threshold_selection": False,
        "test_used_for_threshold_selection": False,
        "warning": (
            "Test scenarios share some fault families with training. "
            "This is not a fully independent unseen-fault-family test."
        )
    }, f, indent=2)

print("\nSaved:")
print(pred_path)
print(metrics_path)
print(scenario_path)
print(threshold_path)


=== TRAIN-ONLY THRESHOLD CALIBRATION ===
Training scenarios: 5
Maximum non-fault training score: 1.3912332343316047e-11
Minimum faulty training score: 10.050378152592137
Selected threshold: 5.0251890763030245

=== CELL-LEVEL METRICS BY SPLIT ===
split  scenarios  TP  FP  FN   TN  precision  recall  specificity  f1
train          5   3   0   0  497        1.0     1.0          1.0 1.0
  val          3   2   0   0  298        1.0     1.0          1.0 1.0
 test         20  26   0   0 1974        1.0     1.0          1.0 1.0

=== SCENARIO-LEVEL RESULTS ===
train: all faults detected in 3/3 fault scenarios
train: healthy scenarios without false alarms 2/2
val: all faults detected in 2/2 fault scenarios
val: healthy scenarios without false alarms 1/1
test: all faults detected in 20/20 fault scenarios

=== TEST SCENARIO DETAILS ===
    scenario_instance_id actual_fault_cells predicted_defective_cells missed_fault_cells false_alarm_cells
        S4_CELL037_BENCH               [37]              

In [33]:

import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/PINN_EV_Battery/"
    "07_V1_AllCell_Localization_EarlyWarning.ipynb"
)

# Find the actual Notebook 07 file if the expected name differs.
if not notebook_path.exists():
    candidates = list(
        Path("/content/drive/MyDrive/PINN_EV_Battery").glob("*.ipynb")
    )
    print("Notebook files found:")
    for p in candidates:
        print(p)
else:
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell.get("cell_type") == "code" and i >= 10:
            source = "".join(cell.get("source", []))
            print(f"\n{'='*20} CELL {i} {'='*20}\n")
            print(source[:8000])


Notebook files found:


In [35]:

# CELL 11 — Figures, coverage, and honest early-warning status
# COMMAND: Initialize and validate the output directory before saving files.

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import json
import torch

# --------------------------------------------------
# 0. FIX OUTPUT DIRECTORY
# --------------------------------------------------
ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")
OUT = ROOT / "results_v1_localization"

# Create the directory if it does not exist.
OUT.mkdir(parents=True, exist_ok=True)

# Validate that OUT is a directory.
if not OUT.is_dir():
    raise NotADirectoryError(
        f"OUT must be a directory, but received: {OUT}"
    )

print("Project root:", ROOT)
print("Output directory:", OUT)
print("Output directory exists:", OUT.is_dir())

# Validate required variables from previous cells.
required_variables = [
    "m", "pred", "scen", "SEED", "DEVICE",
    "CHECKPOINT", "SCALER", "INPUT_COLS",
    "TARGET_COL", "HORIZONS"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise NameError(
        "Run the prerequisite cells first. Missing variables: "
        + ", ".join(missing_variables)
    )

# --------------------------------------------------
# 1. Fault-rank distribution
# --------------------------------------------------
if "mean_observed_fault_rank" not in m.columns:
    raise KeyError(
        "Column 'mean_observed_fault_rank' is missing from m. "
        "Check the original localization metrics cell."
    )

valid_ranks = pd.to_numeric(
    m["mean_observed_fault_rank"], errors="coerce"
).dropna()

plt.figure(figsize=(8, 5))

if len(valid_ranks):
    plt.hist(
        valid_ranks,
        bins=min(15, max(1, len(valid_ranks)))
    )
else:
    plt.text(
        0.5, 0.5,
        "No observed fault ranks available",
        ha="center", va="center"
    )

plt.xlabel("Mean rank of observed ground-truth fault cell(s)")
plt.ylabel("Scenario-instance × horizon evaluations")
plt.title("Exploratory PINN localization: fault-cell rank")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(
    OUT / "07_fault_rank_distribution.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

print("Saved:", OUT / "07_fault_rank_distribution.png")

# --------------------------------------------------
# 2. Recall by scenario instance and horizon
# --------------------------------------------------
if not {"horizon_s", "recall_at_k"}.issubset(m.columns):
    raise KeyError(
        "m must contain 'horizon_s' and 'recall_at_k'."
    )

plt.figure(figsize=(8, 5))

for h, group in m.groupby("horizon_s"):
    group = group.reset_index(drop=True)

    plt.plot(
        np.arange(len(group)),
        pd.to_numeric(
            group["recall_at_k"], errors="coerce"
        ).to_numpy(),
        marker="o",
        label=f"{h}s"
    )

plt.xlabel("Scenario-instance index within horizon")
plt.ylabel("Recall@k")
plt.title("Horizon-specific PINN localization recall")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(
    OUT / "07_recall_by_instance.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

print("Saved:", OUT / "07_recall_by_instance.png")

# --------------------------------------------------
# 3. Honest early-warning evaluation
# --------------------------------------------------
scenario_cols = set(scen.columns)

onset_candidates = [
    "fault_onset_time_s",
    "fault_onset_s",
    "fault_start_time_s",
    "fault_start_s",
    "fault_time_s",
    "onset_time_s"
]

onset_col = next(
    (c for c in onset_candidates if c in scenario_cols),
    None
)

early_warning = {
    "status": (
        "NOT_EVALUABLE_MISSING_FAULT_ONSET_TIMESTAMP"
        if onset_col is None
        else "ONSET_COLUMN_FOUND_REQUIRES_VALIDATION"
    ),
    "onset_column": onset_col,
    "lead_time_seconds_calculated": False,
    "reason": (
        "Scenario labels identify fault cells but do not contain "
        "a recognized fault-onset timestamp. Prediction horizons "
        "(60/300/600 s) are not themselves measured warning lead times."
        if onset_col is None
        else
        "An onset-like column exists; its semantics and units must "
        "be validated before computing lead time."
    )
}

(OUT / "07_early_warning_evaluation.json").write_text(
    json.dumps(early_warning, indent=2),
    encoding="utf-8"
)

# --------------------------------------------------
# 4. Read Cell 9 coverage audit
# --------------------------------------------------
coverage_path = OUT / "07_cell_coverage_audit.json"

if coverage_path.exists():
    with open(coverage_path, "r", encoding="utf-8") as f:
        coverage = json.load(f)
else:
    coverage = {}
    print(
        "WARNING: Cell 9 coverage audit not found. "
        "Coverage cannot be confirmed from the audit file."
    )

all_100_cells_from_audit = bool(
    coverage.get(
        "all_100_cells_in_predictions",
        coverage.get(
            "all_100_cells_scored",
            coverage.get("all_100_cells_present", False)
        )
    )
)

prediction_cell_ids = (
    pd.to_numeric(pred["cell_id"], errors="coerce")
    .dropna()
    .astype(int)
    .unique()
)

prediction_cell_count = len(prediction_cell_ids)

prediction_has_all_100 = (
    prediction_cell_count == 100
    and set(prediction_cell_ids) == set(range(1, 101))
)

all_100_cells_present = (
    all_100_cells_from_audit and prediction_has_all_100
)

print("\nCell IDs found in PINN prediction dataframe:",
      prediction_cell_count)
print("Audit reports full coverage:", all_100_cells_from_audit)
print("Prediction dataframe covers cells 1–100:",
      prediction_has_all_100)

# --------------------------------------------------
# 5. Safe metric calculations
# --------------------------------------------------
def safe_mean(series):
    values = pd.to_numeric(
        series, errors="coerce"
    ).dropna()

    return float(values.mean()) if len(values) else None

# --------------------------------------------------
# 6. Results summary — ORIGINAL PINN LOCALIZATION PATH
# --------------------------------------------------
summary = {
    "notebook": (
        "07_V1_AllCell_Localization_EarlyWarning_CORRECTED.ipynb"
    ),
    "evaluation_path": "PINN_future_SOH_localization",
    "status": "EXPLORATORY_NOT_SAFETY_VALIDATED",
    "seed": int(SEED),
    "device": str(DEVICE),
    "torch_version": torch.__version__,
    "checkpoint": str(CHECKPOINT),
    "scaler": str(SCALER),
    "input_columns": list(INPUT_COLS),
    "target": TARGET_COL,
    "horizons_s": sorted(int(h) for h in HORIZONS),

    "test_instances_scored": int(
        pred["scenario_instance_id"].nunique()
    ),
    "cells_scored": int(prediction_cell_count),
    "all_100_cells_present": bool(all_100_cells_present),

    "instances_evaluated": int(
        m["scenario_instance_id"].nunique()
    ),
    "evaluations_instance_by_horizon": int(len(m)),

    "mean_precision_at_k": safe_mean(m["precision_at_k"]),
    "mean_recall_at_k": safe_mean(m["recall_at_k"]),
    "mean_f1_at_k": safe_mean(m["f1_at_k"]),
    "mean_observed_fault_rank": safe_mean(
        m["mean_observed_fault_rank"]
    ),

    "fault_labels_used_as_model_inputs": False,
    "early_warning_lead_time_calculated": False,
    "early_warning_status": early_warning["status"],

    "limitations": [
        "Absolute deviation from cohort median is an exploratory "
        "ranking heuristic, not a validated fault classifier.",
        "These metrics describe the PINN localization path, not "
        "the separate signal-statistics threshold detector.",
        "Localization metrics are computed separately for each horizon.",
        "Missing cell IDs and unobserved fault cells are reported, "
        "not fabricated.",
        "Early-warning lead time requires valid fault-onset timestamps "
        "and is not inferred from prediction horizon.",
        "Perfect signal-based threshold metrics must not be substituted "
        "for PINN localization metrics."
    ]
}

summary_path = OUT / "07_results_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n=== PINN RESULTS SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\n=== CELL COVERAGE AUDIT ===")
print(json.dumps(coverage, indent=2))

print("\n=== EARLY-WARNING AUDIT ===")
print(json.dumps(early_warning, indent=2))

print("\nSaved results to:", summary_path)
print("\nCELL 11 COMPLETED SUCCESSFULLY.")


Project root: /content/drive/MyDrive/PINN_EV_Battery
Output directory: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization
Output directory exists: True
Saved: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_fault_rank_distribution.png
Saved: /content/drive/MyDrive/PINN_EV_Battery/results_v1_localization/07_recall_by_instance.png

Cell IDs found in PINN prediction dataframe: 100
Audit reports full coverage: True
Prediction dataframe covers cells 1–100: True

=== PINN RESULTS SUMMARY ===
{
  "notebook": "07_V1_AllCell_Localization_EarlyWarning_CORRECTED.ipynb",
  "evaluation_path": "PINN_future_SOH_localization",
  "status": "EXPLORATORY_NOT_SAFETY_VALIDATED",
  "seed": 20261012,
  "device": "cuda",
  "torch_version": "2.11.0+cu130",
  "checkpoint": "/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/models_v2/B3_physics_domain_v2_best.pt",
  "scaler": "/content/drive/MyDrive/PINN_EV_Battery/data/v1/results_v1/06_training_normalization_v2.json",
  "in

In [37]:
# CELL 12 — Final engineering and research-requirements gate (no hardcoded PASS)
required = [
    "07_artifact_inventory.csv",
    "07_checkpoint_inspection.json",
    "07_normalization_inspection.json",
    "07_dataset_schema_audit.json",
    "07_split_isolation_audit.json",
    "07_cell_predictions.csv",
    "07_cell_coverage_audit.json",
    "07_localization_scores.csv",
    "07_localization_metrics.csv",
    "07_localization_predictions.csv",
    "07_healthy_case_false_localization.csv",
    "07_early_warning_evaluation.json",
    "07_results_summary.json",
    "07_fault_rank_distribution.png",
    "07_recall_by_instance.png"
]

checkpoint_report_path = OUT / "07_checkpoint_inspection.json"
normalization_report_path = OUT / "07_normalization_inspection.json"
split_report_path = OUT / "07_split_isolation_audit.json"
coverage_report_path = OUT / "07_cell_coverage_audit.json"
early_warning_path = OUT / "07_early_warning_evaluation.json"

checkpoint_report = json.loads(checkpoint_report_path.read_text()) if checkpoint_report_path.exists() else {}
normalization_reports = json.loads(normalization_report_path.read_text()) if normalization_report_path.exists() else []
split_report = json.loads(split_report_path.read_text()) if split_report_path.exists() else {}
coverage_report = json.loads(coverage_report_path.read_text()) if coverage_report_path.exists() else {}
early_warning_report = json.loads(early_warning_path.read_text()) if early_warning_path.exists() else {}

checks = {
    "input_contract_8": INPUT_COLS == [
        "cell_voltage_V", "cell_current_A", "cell_temperature_K",
        "surface_temperature_K", "soc_proxy", "requested_charge_rate_C",
        "ambient_temperature_C", "horizon_s"
    ],
    "target_contract": TARGET_COL == "capacity_retention_soh__future",
    "horizon_coverage": HORIZONS == {60, 300, 600},
    "checkpoint_compatible": (
        checkpoint_report.get("dimensions", {}).get("input_dim") == 8
        and checkpoint_report.get("dimensions", {}).get("output_dim") == 1
        and checkpoint_report.get("input_columns") == INPUT_COLS
        and checkpoint_report.get("output_columns") == [TARGET_COL]
    ),
    "normalization_compatible": (
        len(normalization_reports) == 1
        and bool(normalization_reports[0].get("compatible"))
    ),
    "split_isolation": bool(split_report.get("isolation", False)),
    "fault_labels_not_inputs": not any("fault" in c.lower() for c in INPUT_COLS),
    "finite_predictions": bool(np.isfinite(pred["predicted_future_soh"].to_numpy()).all()),
    "horizon_specific_metrics": set(m["horizon_s"].astype(int).unique()) == HORIZONS,
    "all_100_cells_present": bool(
    coverage_report.get(
        "all_100_cells_in_predictions",
        coverage_report.get(
            "all_100_cells_scored",
            coverage_report.get("all_100_cells_present", False)
        )
    )
),
    "early_warning_lead_time_evaluable": (
        early_warning_report.get("status") == "LEAD_TIME_EVALUATED"
    ),
    "required_outputs_exist": all(
        (OUT / name).exists() and (OUT / name).stat().st_size > 0
        for name in required
    )
}

engineering_checks = [
    "input_contract_8", "target_contract", "horizon_coverage",
    "checkpoint_compatible", "normalization_compatible", "split_isolation",
    "fault_labels_not_inputs", "finite_predictions", "horizon_specific_metrics",
    "required_outputs_exist"
]
engineering_pass = all(checks[k] for k in engineering_checks)
research_requirements_pass = engineering_pass and checks["all_100_cells_present"] and checks["early_warning_lead_time_evaluable"]

gate = {
    "engineering_status": "PASS" if engineering_pass else "FAIL",
    "research_requirements_status": "PASS" if research_requirements_pass else "NOT_SATISFIED",
    "checks": checks,
    "mean_precision_at_k": float(m["precision_at_k"].mean()) if m["precision_at_k"].notna().any() else None,
    "mean_recall_at_k": float(m["recall_at_k"].mean()) if m["recall_at_k"].notna().any() else None,
    "mean_f1_at_k": float(m["f1_at_k"].mean()) if m["f1_at_k"].notna().any() else None,
    "warning": (
        "Engineering PASS means the pipeline ran and its artifacts passed structural checks. "
        "It does not establish safety validity. Research requirements remain unsatisfied when "
        "the source data lacks the expected 100 cells or valid fault-onset timestamps."
    )
}
(OUT / "07_final_gate.json").write_text(json.dumps(gate, indent=2), encoding="utf-8")
print(json.dumps(gate, indent=2))
if not engineering_pass:
    raise RuntimeError("ENGINEERING GATE FAILED. Inspect 07_final_gate.json.")
if not research_requirements_pass:
    print(
        "\nNOTE: ENGINEERING PIPELINE RAN, BUT RESEARCH REQUIREMENTS ARE NOT SATISFIED. "
        "Do not claim 100-cell localization or validated early-warning lead time."
    )
else:
    print("\nNOTEBOOK 07 ENGINEERING AND RESEARCH REQUIREMENTS: PASS")


{
  "engineering_status": "PASS",
  "research_requirements_status": "NOT_SATISFIED",
  "checks": {
    "input_contract_8": true,
    "target_contract": true,
    "horizon_coverage": true,
    "checkpoint_compatible": true,
    "normalization_compatible": true,
    "split_isolation": true,
    "fault_labels_not_inputs": true,
    "finite_predictions": true,
    "horizon_specific_metrics": true,
    "all_100_cells_present": true,
    "early_warning_lead_time_evaluable": false,
    "required_outputs_exist": true
  },
  "mean_precision_at_k": 0.0,
  "mean_recall_at_k": 0.0,
  "mean_f1_at_k": 0.0,
  "warning": "Engineering PASS means the pipeline ran and its artifacts passed structural checks. It does not establish safety validity. Research requirements remain unsatisfied when the source data lacks the expected 100 cells or valid fault-onset timestamps."
}

NOTE: ENGINEERING PIPELINE RAN, BUT RESEARCH REQUIREMENTS ARE NOT SATISFIED. Do not claim 100-cell localization or validated early-warn

In [38]:
#Cell 13 — Final signal-based detector report
from pathlib import Path
import json
import pandas as pd

ROOT = Path("/content/drive/MyDrive/PINN_EV_Battery")
OUT = ROOT / "results_v1_localization"

metrics_path = OUT / "07_threshold_v3_metrics_by_split.csv"
scenarios_path = OUT / "07_threshold_v3_scenario_results.csv"
calibration_path = OUT / "07_threshold_v3_calibration.json"

for path in [metrics_path, scenarios_path, calibration_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required output missing: {path}")

metrics = pd.read_csv(metrics_path)
scenarios = pd.read_csv(scenarios_path)

with open(calibration_path, "r", encoding="utf-8") as f:
    calibration = json.load(f)

test_metrics = metrics[metrics["split"] == "test"]
if len(test_metrics) != 1:
    raise ValueError("Expected exactly one test-split metrics row.")

test = test_metrics.iloc[0]

test_scenarios = scenarios[scenarios["split"] == "test"]
fault_test = test_scenarios[~test_scenarios["healthy_scenario"]]
healthy_test = test_scenarios[test_scenarios["healthy_scenario"]]

summary = {
    "detector": "Signal-statistics anomaly detector",
    "status": "EXPLORATORY_HELD_OUT_INSTANCE_EVALUATION",
    "score_source": "07_signal_localization_diagnostic.csv",
    "threshold": calibration["threshold"],
    "threshold_selection_method": calibration["selection_method"],
    "threshold_calibration_split": calibration["calibration_split"],

    "test_cell_level": {
        "TP": int(test["TP"]),
        "FP": int(test["FP"]),
        "FN": int(test["FN"]),
        "TN": int(test["TN"]),
        "precision": float(test["precision"]),
        "recall": float(test["recall"]),
        "specificity": float(test["specificity"]),
        "f1": float(test["f1"])
    },

    "test_scenario_level": {
        "total_scenarios": int(len(test_scenarios)),
        "fault_scenarios": int(len(fault_test)),
        "fault_scenarios_with_all_faults_detected": int(
            fault_test["all_faults_detected"].sum()
        ),
        "healthy_scenarios": int(len(healthy_test)),
        "healthy_scenarios_without_false_alarms": int(
            healthy_test["healthy_false_alarm_free"].sum()
        )
    },

    "limitations": [
        "Results apply to this generated dataset and its current scenario splits.",
        "Test scenarios include charge-rate variations of existing fault families.",
        "Performance on completely unseen fault families has not been established.",
        "The signal-statistics detector is separate from the trained PINN SOH predictor.",
        "Early-warning lead time is not validated without fault-onset timestamps.",
        "These results do not establish real-world battery safety or deployment readiness."
    ]
}

output_path = OUT / "07_signal_detector_final_summary.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("=== SIGNAL-BASED DETECTOR: HELD-OUT TEST RESULTS ===")
print(json.dumps(summary, indent=2))
print("\nSaved:", output_path)


=== SIGNAL-BASED DETECTOR: HELD-OUT TEST RESULTS ===
{
  "detector": "Signal-statistics anomaly detector",
  "status": "EXPLORATORY_HELD_OUT_INSTANCE_EVALUATION",
  "score_source": "07_signal_localization_diagnostic.csv",
  "threshold": 5.0251890763030245,
  "threshold_selection_method": "midpoint between maximum non-fault and minimum faulty training scores",
  "threshold_calibration_split": "train",
  "test_cell_level": {
    "TP": 26,
    "FP": 0,
    "FN": 0,
    "TN": 1974,
    "precision": 1.0,
    "recall": 1.0,
    "specificity": 1.0,
    "f1": 1.0
  },
  "test_scenario_level": {
    "total_scenarios": 20,
    "fault_scenarios": 20,
    "fault_scenarios_with_all_faults_detected": 20,
    "healthy_scenarios": 0,
    "healthy_scenarios_without_false_alarms": 0
  },
  "limitations": [
    "Results apply to this generated dataset and its current scenario splits.",
    "Test scenarios include charge-rate variations of existing fault families.",
    "Performance on completely unseen f